# 02 Data Cleaning

In [1]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
import trafilatura
import matplotlib.pyplot as plt
import seaborn as sns
import html
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import sys
import json

In [2]:
df = pd.read_parquet("../data/processed/articles_clean_notebook.parquet")

## 1. Field Quality: `full_text` vs `summary`

We have two text fields. Understanding their relationship helps decide which one to use downstream.

In [47]:
# Compare lengths
df["full_text_len"] = df["full_text"].replace("", pd.NA).str.len()
df["summary_len"] = df["summary"].replace("", pd.NA).str.len()

stats = pd.concat(
    [
        df["full_text_len"].describe().round(0).rename("full_text"),
        df["summary_len"].describe().round(0).rename("summary"),
    ],
    axis=1,
)
print(stats.to_string())

       full_text  summary
count    51801.0  51622.0
mean      2565.0    818.0
std       3219.0   1317.0
min         95.0      4.0
25%       1170.0    123.0
50%       1849.0    507.0
75%       2932.0   1003.0
max      83416.0  31422.0


In [48]:
# How often is full_text shorter than or equal to summary?
full_shorter = df[df["full_text_len"] <= df["summary_len"]]
print(
    f"Articles where full_text <= summary in length: {len(full_shorter)} ({len(full_shorter)/len(df)*100:.1f}%)"
)
print()
print(full_shorter[["source", "full_text_len", "summary_len"]].value_counts("source"))

Articles where full_text <= summary in length: 4646 (9.0%)

source
fakti      3140
24chasa    1019
banker      216
dnevnik     162
bta         109
Name: count, dtype: int64


In [49]:
for field in ["full_text", "summary"]:
    html_mask = df[field].fillna("").str.contains(r"<[a-zA-Z]", regex=True)
    print(f"=== {field.upper()} ===")
    print(f"Articles with HTML: {html_mask.sum()} ({html_mask.sum()/len(df)*100:.1f}%)")
    print("By source:")
    print(df[html_mask]["source"].value_counts().to_string())
    print()
    for source in df[html_mask]["source"].unique():
        sample = df[html_mask & (df["source"] == source)][field].iloc[0]
        print(f"--- {source} ---")
        print(sample[:400])
        print()
    print("=" * 60)
    print()

=== FULL_TEXT ===
Articles with HTML: 3 (0.0%)
By source:
source
monitor     2
actualno    1

--- monitor ---
Е дин човек е загинал, а най-малко четирима са ранени при мощното земетресение на филипинския остров Минданао, предаде Франс прес. Срутили са се и няколко сгради на различни места в архипелага, уточни полицията.
Earthquake in General Santos City, The Philippines on June 8, 2026 pic.twitter.com/ye8LHYwVzB
— Noypi (@noypistuff) June 8, 2026
"Няколко сгради са рухнали. Няколко къщи също са се срутили

--- actualno ---
Ориентираме се към край на сериозните новини за днес! Време е за вечерната ни доза допамин – няколко секунди, които да ни напомнят, че светът е пълен с причини за усмивка:
<blockquote class="twitter-tweet"><p lang="ja" dir="ltr">カワウソの寝る前のルーティン<a href="https://t.co/LWQr1lUDka">pic.twitter.com/LWQr1lUDka</a></p>— 癒される動物 (@cutest_animal1) <a href="https://x.com/cutest_animal1/status/20628451592940830


=== SUMMARY ===
Articles with HTML: 21963 (42.4%)
By source:
source


- full_text is on average 3.4× longer than summary, confirming it as the primary field for embedding and retrieval.
- ~50% of summaries contain raw HTML tags — stripping is required before any use.
- ~10% of articles have full_text ≤ summary length, concentrated in fakti (likely HTML inflation) and 24chasa/dnevnik (possibly truncated full text).

### 1a. HTML striping from `summary`

In [50]:
def strip_html(text):
    if not isinstance(text, str):
        return ""
    return BeautifulSoup(text, "html.parser").get_text(separator=" ", strip=True)

In [51]:
def print_by_words(text, chunk=20):
    words = text.split()
    for i in range(0, len(words), chunk):
        print(" ".join(words[i : i + chunk]))

In [52]:
for _, row in df[df["summary"].str.len() > 0].sample(3, random_state=42).iterrows():
    print(f"source: {row['source']}")
    print_by_words(row["summary"])
    print()

source: 24chasa
<img alt="Осъдиха на 6 години затвор и 15 000 евро глоба 44-годишен софиянец, продавал фентанил" height="600" src="https://cache2.24chasa.bg/Images/Cache/010/IMAGE_22705010_2_0.jpg" width="800" /> Шест
години лишаване от свобода и глоба от 15 хил. евро постанови Окръжният съд в Търговище за 44-годишен мъж от София
за държане с цел разпространение на фентанил, съобщиха от съда. С. В. от София (44-годишен, осъждан) бе признат за виновен
в това, че на 24.06.2025 г. е държал с цел разпространение високорискови наркотични вещества

source: standartnews
На 95 години почина Сони Ролинс, смятан за един от последните велики имена на златната ера на джаза и известен
като „Саксофонният колос&quot;.„С дълбока скръб и огромна любов съобщаваме за кончината на Сони Ролинс&quot;, гласи публикацията в официалната му страница
в социалните мрежи.Ролинс е роден в Ню Йорк. Първият си саксофон получава, когато е на 13. Ролинс отначало е пианист,
после преминава към алто саксофона, и в крайна

In [53]:
df["summary"] = df["summary"].apply(strip_html)

In [54]:
for _, row in df[df["summary"].str.len() > 0].sample(3).iterrows():
    print(f"source: {row['source']}")
    print_by_words(row["summary"])
    print()

source: fakti
Приключи първата репетиция на България на сцената на „Евровизия 2026“ във Виена, а около участието на DARA вече се завихря
сериозен интерес и положителни коментари от медии, следящи отблизо конкурса. Още от първите минути на сцената българското представяне е успяло
да привлече внимание, а информацията, която излиза след техническата репетиция, само подсилва очакванията към втория полуфинал, пише darik.bg. DARA е
провела първата си техническа репетиция на огромната сцена във Виена и според присъстващи и наблюдатели се е раздала напълно –
както вокално, така и сценично. Изпълнението ѝ е описвано като енергично, уверено и добре структурирано, с ясно изразено сценично присъствие,
което не остава незабелязано дори сред силната конкуренция тази година. Още по-интересен обаче е фактът, който вече се коментира от
медии като Eurovisionfun и Eurovision Cloud, България е първата страна тази година, която прави промени по своята песен след началото
на репетициите. Според информацията

In [55]:
for field in ["summary"]:
    html_mask = df[field].fillna("").str.contains(r"<[a-zA-Z]", regex=True)
    print(f"=== {field.upper()} ===")
    print(f"Articles with HTML: {html_mask.sum()} ({html_mask.sum()/len(df)*100:.1f}%)")
    print("By source:")
    print(df[html_mask]["source"].value_counts().to_string())
    print()
    for source in df[html_mask]["source"].unique():
        sample = df[html_mask & (df["source"] == source)][field].iloc[0]
        print(f"--- {source} ---")
        print_by_words(sample)
        print()
    print("=" * 60)
    print()

=== SUMMARY ===
Articles with HTML: 282 (0.5%)
By source:
source
fakti    282

--- fakti ---
Бразилският защитник на Левски Майкон официално се ожени за своята дългогодишна половинка Тайнара Гомес. Двойката отпразнува своя специален ден с
пищна церемония, а щастливите мигове не закъсняха да се появят в социалните мрежи, където приятели, фенове и съотборници се надпреварваха
да ги поздравят. <blockquote class="instagram-media" data-instgrm-captioned data-instgrm-permalink="https://www.instagram.com/p/DZSv9W8jix-/?utm_source=ig_embed&amp;utm_campaign=loading" data-instgrm-version="14" style=" background:#FFF; border:0; border-radius:3px; box-shadow:0 0 1px 0 rgba(0,0,0,0.5),0 1px 10px 0
rgba(0,0,0,0.15); margin: 1px; max-width:540px; min-width:326px; padding:0; width:99.375%; width:-webkit-calc(100% - 2px); width:calc(100% - 2px);"><div style="padding:16px;"> <a href="https://www.instagram.com/p/DZSv9W8jix-/?utm_source=ig_embed&amp;utm_campaign=loading" style=" background:#FFFFFF; line-h

## Decision: Drop `summary`

The `summary` field is sourced directly from the RSS `<description>` tag and is inconsistent across sources — some articles have a meaningful short intro, others have a duplicate of `full_text`, and roughly 50% contain raw HTML markup. Several sources (`vesti`, `segabg`) provide no summary at all.

Since `full_text` is the authoritative text field used for all downstream tasks `summary` adds no value and is dropped.


In [56]:
df = df.drop(columns=["summary", "full_text_len", "summary_len"])

In [57]:
df.head(3)

,source,title,url,published_at,full_text,fetched_at,published_at_dt,word_count
0,vesti,<p>Ким Чен-ун и Си Дзинпин с ключова среща в П...,https://www.vesti.bg/sviat/kim-chen-un-i-si-dz...,2026-06-09T07:35:00+00:00,С евернокорейският лидер Ким Чен-ун и китайски...,2026-06-09T11:20:07.411349+00:00,2026-06-09 07:35:00+00:00,315
1,vesti,<p>Зеленски с &bdquo;желязна&ldquo; подкрепа о...,https://www.vesti.bg/sviat/zelenski-s-zheliazn...,2026-06-09T08:04:00+00:00,У краинският президент Володимир Зеленски се с...,2026-06-09T11:20:07.411341+00:00,2026-06-09 08:04:00+00:00,396
2,vesti,New York Times: Израел сипе бял фосфор над гра...,https://www.vesti.bg/sviat/new-york-times-izra...,2026-06-09T08:05:00+00:00,"В изуални доказателства, събрани и проверени о...",2026-06-09T11:20:07.411332+00:00,2026-06-09 08:05:00+00:00,1103


## 2. Cyrillic & Encoding Quality Check

Bulgarian uses the Cyrillic alphabet. Encoding issues (e.g. Windows-1251 misread as UTF-8) produce garbled text.
We check how much of the text is valid Cyrillic and flag suspicious articles.

In [58]:
def is_mojibake(text: str) -> bool:
    if not isinstance(text, str) or not text.strip():
        return False
    dash_ratio = (text.count("–") + text.count("—")) / max(len(text), 1)
    return dash_ratio > 0.05

In [59]:
mojibake_mask = df["full_text"].apply(is_mojibake)
print(
    f"Unreadable (mojibake): {mojibake_mask.sum()} ({mojibake_mask.sum()/len(df)*100:.1f}%)"
)
print(df[mojibake_mask]["source"].value_counts().to_string())
print()

Unreadable (mojibake): 62 (0.1%)
source
actualno    62



In [60]:
for _, row in df[mojibake_mask][["source", "title", "full_text"]].head(3).iterrows():
    print(f"  Source : {row['source']}")
    print(f"  Title  : {row['title']}")
    print(f"  Preview: {row['full_text'][:200]}")
    print()

  Source : actualno
  Title  : Оръжията да говорят: Лавров призова, Украйна изпълни серия унищожителни удари (ОБЗОР - ВИДЕО)
  Preview: "–Ь–љ–Њ–≥–Њ –њ–Њ–Ј–Є—В–Є–≤–µ–љ" - —В–∞–Ї—К–≤ –µ –±–Є–ї —А–∞–Ј–≥–Њ–≤–Њ—А—К—В –Љ–µ–ґ–і—Г —Г–Ї—А–∞–Є–љ—Б–Ї–Є—П –њ—А–µ–Ј–Є–і–µ–љ—В –Т–Њ–ї–Њ–і–Є–Љ–Є—А –Ч–µ–ї–µ–љ—Б–Ї–Є, —Б–њ–µ—Ж–Є–∞–ї–љ–Є—П –њ—А–∞—В–µ–љ–Є–

  Source : actualno
  Title  : Проектът на зетя на Тръмп в Албания е "благословия": Албанците не са съгласни с Еди Рама и са по улиците (ВИДЕО)
  Preview: –Ю—Б—В–∞–≤–Ї–∞ –љ–∞ –њ—А–µ–Љ–Є–µ—А–∞ –Х–і–Є –†–∞–Љ–∞ - –≤ —В–Њ–≤–∞ –≤–µ—З–µ —Б–µ –њ—А–µ–≤—К—А–љ–∞—Е–∞ –њ—А–Њ—В–µ—Б—В–Є—В–µ —Б—А–µ—Й—Г –Љ–∞—Й–∞–±–љ–Є—П –Є–љ–≤–µ—Б—В–Є—Ж–Є–Њ–љ–µ–љ –њ—А–Њ–µ–Ї—В –≤ –Р–ї–±

  Source : actualno
  Title  : След като Армения избра Европа, Русия я обвини в "груби нарушения на демокрацията" (ВИДЕО)
  Preview: –†―É―¹–Η―è –¥–Α ―²–Η –Η–Ζ–Ϋ–Α―¹―è ―É―Ä–Ψ–Κ –Ω–Ψ –¥–Β–Φ–Ψ–Κ―Ä–Α―Ü–Η―è –Η –¥–Β–Φ–Ψ–Κ―Ä–Α―²–Η―΅–Ϋ–Η ―Ü–Β–Ϋ–Ϋ–Ψ―¹―²–Η - –Η –¥–Ψ ―²–Ψ–≤–Α ―¹―²–Η–≥–Ϋ–Α―Ö–Φ–Β –Ω―Ä–Β–Ζ 2026 –≥., ―

In [61]:
df = df[~mojibake_mask].reset_index(drop=True)
print(f"Dropped {mojibake_mask.sum()} unreadable articles. Remaining: {len(df)}")

Dropped 62 unreadable articles. Remaining: 51739


**Finding:** A small number of articles from `actualno` contained fully unreadable text — classic mojibake produced by Windows-1251 content misread as UTF-8, recognisable by an abnormally high density of `–`/`—` characters.

**Decision:** Dropped all articles where more than 5% of characters are em/en dashes. All remaining articles contain readable Bulgarian text.


## 3. Source-Specific Investigations

### 3a. Dnevnik — JavaScript-Rendered Content

In [62]:
if "dnevnik" in df["source"].values:
    dnevnik_sample = df[df["source"] == "dnevnik"][
        ["title", "url", "word_count", "full_text"]
    ].sort_values("word_count")
    print(f"Total dnevnik articles : {len(dnevnik_sample)}")
    print(f"Median word count      : {dnevnik_sample['word_count'].median()}")
    print()
    for _, row in dnevnik_sample.head(3).iterrows():
        print(f"Title   : {row['title']}")
        print(f"Words   : {row['word_count']}")
        print(f"Full text : {row['full_text'][:300]}")
        print("-" * 60)

Total dnevnik articles : 255
Median word count      : 23.0

Title   : Слънчево и топло време в събота
Words   : 13
Full text : 06:20Слънчево и топло време в съботаДневник0СподелиЛинкът е копиранКопирай линкИзпрати по имейлFacebookХLinkedInViber25 април 2026
------------------------------------------------------------
Title   : САЩ разхлабват правилата за канабиса
Words   : 14
Full text : 21:30САЩ разхлабват правилата за канабисаДневник5СподелиЛинкът е копиранКопирай линкИзпрати по имейлFacebookХLinkedInViber23 април 2026ВСИЧКИ БЪРЗИ НОВИНИ
------------------------------------------------------------
Title   : "Левски" има нов собственик
Words   : 14
Full text : HomeСпорт"Левски" има нов собственикЦВЕТЕЛИНА БЕЛУТОВААтанас БостанджиевАндриян Георгиев37СподелиЛинкът е копиранКопирай линкИзпрати по имейлFacebookХLinkedInViber24 април 2026
------------------------------------------------------------


In [63]:
# Verify with trafilatura directly
if "dnevnik" in df["source"].values:
    test_url = df[df["source"] == "dnevnik"]["url"].iloc[0]
    downloaded = trafilatura.fetch_url(test_url)
    result = trafilatura.extract(downloaded) if downloaded else None
    print(
        f"Trafilatura extraction result: {repr(result[:200]) if result else 'None — extraction failed'}"
    )

Trafilatura extraction result: 'HomeСвятПреговорите САЩ-Иран в Пакистан приключиха преди да започнат, Тръмп ги прекъснаReutersПетър Карабоев97СподелиЛинкът е копиранКопирай линкИзпрати по имейлFacebookХLinkedInViber25 април 2026Удар'


**Finding:** Dnevnik runs on Next.js. Trafilatura receives the bare HTML shell before JavaScript renders the content.
All 253 articles contain only 13–14 words of boilerplate (timestamps, share buttons).

**Decision:** Remove Dnevnik from the dataset.

In [64]:
mask = df["source"] == "dnevnik"

print(f"Dropping {mask.sum()} articles from: dnevnik")
df = df[~mask].reset_index(drop=True)
print(f"Remaining articles: {len(df)}")

Dropping 255 articles from: dnevnik
Remaining articles: 51484


### 3b. 24chasa — Very Long Outliers

In [65]:
chasa_outliers = df[(df["source"] == "24chasa") & (df["word_count"] > 3000)][
    ["title", "url", "word_count", "full_text"]
].sort_values("word_count", ascending=False)

print(f"24chasa articles over 3000 words: {len(chasa_outliers)}")
print()
for _, row in chasa_outliers.head(5).iterrows():
    print(f"Title   : {row['title']}")
    print(f"Words   : {row['word_count']}")
    print(f"Preview : {row['full_text'][:300]}")
    print("-" * 60)

24chasa articles over 3000 words: 55

Title   : Вижте какво е състоянието на пътищата тази вечер
Words   : 13773
Preview : Агенция "Пътна инфраструктура" информира за състоянието на пътищата, въведените ограничения и метеорологичната обстановка у нас към 17:30 часа.
I. МЕТЕОРОЛОГИЧНА ОБСТАНОВКА:
През следващото денонощие синоптичната обстановка в страната ще бъде усложнена. Ще бъде облачно и с повсеместни валежи от дъжд
------------------------------------------------------------
Title   : Вижте какво е състоянието на пътищата тази вечер
Words   : 13621
Preview : Информация за състоянието на републиканските пътища към 17:30 часа на 08.06.2026 г.
МЕТЕОРОЛОГИЧНА ОБСТАНОВКА:
През нощта валежите ще спрат и облачността ще се разкъса и ще намалее до предимно ясно. Ще бъде почти тихо и след полунощ на отделни места в равнинните райони ще се образува мъгла или ниска
------------------------------------------------------------
Title   : Вижте какво е състоянието на пътищата тази вечер
Words   :

In [66]:
short_24chasa = df[(df["source"] == "24chasa") & (df["word_count"] < 50)]
print(f"24chasa articles under 50 words: {len(short_24chasa)}")
print(short_24chasa[["title", "word_count", "full_text"]].to_string())

24chasa articles under 50 words: 193
                                                                                                                title  word_count                                                                                                                                                                                                                                                                                                                                                              full_text
251                                                                     Малкият Иванчо се учи как започва глаголицата          16                                                                                                                                                                                                            https://www.24chasa.bg/mneniya/article/23002668\nwww.24chasa.bg\nМалкият Иванчо се учи как започва глаголицата\n1856\nПоследвайте ни в Google

**Finding:** 24chasa — Two edge cases

Very short articles (< 50 words): 

Inspection shows two patterns:

- Pure boilerplate stubs — newsletter prompts, social links, no real content (e.g. "24 часа в мейла ти...")
- Legitimate short-form content — breaking news flashes, traffic updates, photo captions (37–49 words)

Very long articles (> 3 000 words):
All are structured road condition reports published daily by the Road Infrastructure Agency under the same title ("Вижте какво е състоянието на пътищата тази вечер"), reaching up to 13 773 words. These are not news articles and would pollute RAG retrieval.

### 3c. Per-Source Shortest & Longest Articles

In [67]:
for source in df["source"].unique():
    source_df = df[df["source"] == source].sort_values("word_count")
    print(f"=== {source.upper()} ===")
    print(
        f"Total: {len(source_df)} | Median words: {source_df['word_count'].median():.0f}"
    )
    print()
    print("-- 2 SHORTEST --")
    for _, row in source_df.head(2).iterrows():
        print(f"  [{row['word_count']} words] {row['title']}")
        print(f"  Preview: {row['full_text'][:150]}")
        print()
    print("-- 2 LONGEST --")
    for _, row in source_df.tail(2).iterrows():
        print(f"  [{row['word_count']} words] {row['title']}")
        print(f"  Preview: {row['full_text'][:150]}")
        print()
    print("=" * 60)
    print()

=== VESTI ===
Total: 3147 | Median words: 340

-- 2 SHORTEST --
  [42 words] Коулрофобия, чиклефобия и атаксофобия: От какво се ужасяват Меган Фокс, Джони Деп и Опра Уинфри?
  Preview: С трахът е нормална човешка емоция, която всички изпитваме, и известните и влиятелните личности не правят изключение. Докато някои звезди страдат от п

  [47 words] Арестуваха кмета на Кърджали
  Preview: К метът на Кърджали Ерол Мюмюн е бил арестуван в четвъртък, научи NOVA от свои източници.
По информация от пресцентъра на Община Кърджали градоначални

-- 2 LONGEST --
  [4081 words] <p>Кабинетът &quot;Радев&quot; е готов: Кои са големите изненади</p>
  Preview: К андидатът за министър-председател Румен Радев, излъчен от парламентарната група на „Прогресивна България“, получи днес от президента Илияна Йотова м

  [4945 words] Мистерията на златния ковчег: Голямата измама в международната търговия с антики
  Preview: П рез ноември 2017 г. френският президент Еманюел Макрон пътува до Обединените арабски е

### 3d. Calculator Articles

In [68]:
calculator_articles = df[
    (df["source"] == "actualno")
    & (df["title"].str.contains("валутен калкулатор", na=False))
]

print(f"Calculator articles: {len(calculator_articles)}")
print(calculator_articles[["title", "word_count", "url"]].to_string())

Calculator articles: 151
                                                                                                     title  word_count                                                                                                                                               url
13819    Турска лира - евро. Колко струва една турска лира към едно евро днес, 28 май /валутен калкулатор/         226    https://www.actualno.com/finance/turska-lira-evro-kolko-struva-edna-turska-lira-kym-edno-evro-dnes-28-maj-valuten-kalkulator-news_2599651.html
13822         Долар - евро. Колко струва един щатски долар към едно евро днес, 28 май /валутен калкулатор/          66       https://www.actualno.com/finance/dolar-evro-kolko-struva-edin-shtatski-dolar-kym-edno-evro-dnes-28-maj-valuten-kalkulator-news_2599652.html
13823      Паунд - евро. Колко струва един британски паунд към едно евро днес, 28 май /валутен калкулатор/          40      https://www.actualno.com/finance/paund-evro-kolko-struva

In [69]:
road_mask = (df["source"] == "24chasa") & (
    df["title"].str.contains("състоянието на пътищата", na=False)
)
print(f"Road condition reports: {road_mask.sum()}")

calc_mask = (df["source"] == "actualno") & (
    df["title"].str.contains("валутен калкулатор", na=False)
)
print(f"Calculator articles: {calc_mask.sum()}")

short_mask = df["word_count"] < 30
print(f"Articles under 30 words: {short_mask.sum()}")

Road condition reports: 37
Calculator articles: 151
Articles under 30 words: 40


**Decision:** drop records with following content:
- Boilerplate stubs (< 30 words, 24chasa) — newsletter prompts and social share links with no real content.
- Daily currency calculators (actualno) — four currency pairs published as articles every day, identified by "калкулатор" in the title.
- Road condition reports (24chasa) — daily government bulletins up to 13 773 words, identified by "състоянието на пътищата" in the title.

In [70]:
print(f"Dropping {short_mask.sum()} articles under 30 words")
df = df[~short_mask].reset_index(drop=True)
print(f"Remaining articles: {len(df)}")

Dropping 40 articles under 30 words
Remaining articles: 51444


In [71]:
mask = (df["source"] == "24chasa") & (
    df["title"].str.contains("състоянието на пътищата", na=False)
)
print(f"Dropping {mask.sum()} road condition reports")
df = df[~mask].reset_index(drop=True)

mask = (df["source"] == "actualno") & (
    df["title"].str.contains("валутен калкулатор", na=False)
)
print(f"Dropping {mask.sum()} calculator articles")
df = df[~mask].reset_index(drop=True)

print(f"Remaining: {len(df)}")

Dropping 37 road condition reports
Dropping 151 calculator articles
Remaining: 51256


### 3e. Fakti outliers

In [72]:
fakti_outliers = df[(df["source"] == "fakti") & (df["word_count"] > 2000)].sort_values(
    "word_count", ascending=False
)

print(f"Fakti articles over 2000 words: {len(fakti_outliers)}")
print()
for _, row in fakti_outliers.head(5).iterrows():
    print(f"Title   : {row['title']}")
    print(f"Words   : {row['word_count']}")
    print(f"Preview : {row['full_text'][:400]}")
    print(f"End     : {row['full_text'][-400:]}")
    print("-" * 60)

Fakti articles over 2000 words: 269

Title   : В Москва! Парадът на победата на Червения площад (ВИДЕО)
Words   : 10868
Preview : Парадът в чест на 81-вата годишнина от победата във Великата отечествена война започна на Червения площад, съобщи ТАСС, цитирана от БТА.
Началото му бе отбелязано с ударите на курантите на Спаската кула. Под оркестровото изпълнение на легендарната композиция "Свещената война" на паветата на Червения площад бяха изнесени държавният флаг на Руската федерация и флагът на Съветския съюз, страната побе
End     : оленето и истината винаги се разминават. Не, че това има значение за безроден глупак!
13:57 09.05.2026
390 Северно Корейския ХОБИТ
До коментар #119 от "Мунчо":
Пратил малко РобиДа ги нагостят със
КУЧЕШКО ВАРЕНО
В Северна Корея
Сабаки нетъ.
13:57 09.05.2026
391 Така е
До коментар #386 от "Себе си описа,":
Безродника няма родина и затова клоака като запада го ползва за евтин отпадък!13:58 09.05.2026
----------------------------------------------------------

**Finding:** The long outliers in `fakti` are likely caused by reader comments being included in the extracted `full_text`. This will be addressed in the source-specific cleaning step, after which the word count distribution will be re-checked.


## 4. `full_text` investigation

In [73]:
sources = df["source"].unique().tolist()
sources

['vesti',
 'monitor',
 'standartnews',
 'segabg',
 'bta',
 'banker',
 'economic',
 'actualno',
 'nova',
 '24chasa',
 'fakti']

In [4]:
def print_source_samples(df, source, n=3, random_state=None):
    source_df = df[df["source"] == source]
    print(f"=== {source.upper()} ({len(source_df)} articles) ===\n")
    for _, row in source_df.sample(
        min(n, len(source_df)), random_state=random_state
    ).iterrows():
        print(f"--- {row['title']} ---")
        print(f"URL: {row['url']}")
        print(row["full_text"])
        print()

In [75]:
def drop_lines(text, predicate):
    if not isinstance(text, str):
        return text
    return "\n".join(line for line in text.split("\n") if not predicate(line))

In [76]:
def truncate_at_sentinel(text: str, sentinels: list[str]) -> str:
    if not isinstance(text, str):
        return text
    text_lower = text.lower()
    for sentinel in sentinels:
        idx = text_lower.find(sentinel.lower())
        if idx != -1:
            text = text[:idx].strip()
    return text

In [77]:
def apply_to_source(df, fn, source=None):
    if source:
        mask = df["source"] == source
        df.loc[mask, "full_text"] = df.loc[mask, "full_text"].apply(fn)
    else:
        df["full_text"] = df["full_text"].apply(fn)
    return df

In [78]:
def fix_split_first_letter(text: str, word_set: set) -> str:
    if not isinstance(text, str):
        return text
    match = re.match(r"^([А-Я]) ([а-я])", text)
    if not match:
        return text
    letter, next_char = match.group(1), match.group(2)
    merged = letter + text[2:].split()[0] if text[2:].split() else ""
    if letter not in "ВС" or merged in word_set:
        return letter + next_char + text[3:]
    return text

In [79]:
EMOJI_RE = re.compile(r"[\U0001F000-\U0001FFFF\U00002600-\U000027BF]")
SOCIAL_ATTR = re.compile(r"\(@\w+\).*\b\d{4}\b")


def is_noise_line(line: str) -> bool:
    stripped = line.strip()
    if not stripped:
        return False

    # remove leading dash/dash+space for content checks
    content = re.sub(r"^[—–-]+\s*", "", stripped)

    # social media attribution: — Handle (@handle) date
    if SOCIAL_ATTR.search(stripped):
        return True

    # lines starting with @handle
    if content.startswith("@"):
        return True

    # emoji heavy lines
    words = content.split()
    if words and sum(1 for w in words if EMOJI_RE.search(w)) / len(words) > 0.3:
        return True

    # entirely Latin (no Cyrillic) — covers plain Latin and — Latin text
    if content and not any("\u0400" <= c <= "\u04ff" for c in content):
        return True

    return False

## 4.a. Vesti

In [80]:
print_source_samples(df, "vesti", n=3, random_state=42)

=== VESTI (3147 articles) ===

--- Рапърът Йонислав Йотов-Тото приключи участието си в Hell’s Kitchen ---
URL: https://www.vesti.bg/lyubopitno/rapyryt-ionislav-iotov-toto-prikliuchi-uchastieto-si-v-hells-kitchen-6258870
О ще един участник се сбогува с възможността да грабне звездната победа в Hell’s Kitchen. Рапърът Йонислав Йотов-Тото не се справи с напрежението на станция “Гарнитури” по време на вечерната резервация и шеф Ангелов сложи край на участието му в надпреварата.
Черните куртки и техните опоненти се впуснаха в забавно и изпълнено с екшън предизвикателство с италиански пици на открито. След спечеленото ценно предимство, претендентите за голямата награда от €50 000 се поздравиха с победата, а фотомоделът Нора Недкова получи оранжева престилка.
Вечерната резервация донесе нов обрат, приготвен от шеф Ангелов. Той обедини звездните готвачи и Черните куртки в един общ отбор, предвождан от номинираната Нора Недкова. Въпреки множеството проблеми с комуникацията, играчите съумяха да 

In [81]:
df = apply_to_source(df, lambda t: drop_lines(t, is_noise_line), source="vesti")

In [82]:
print_source_samples(df, "vesti", n=3)

=== VESTI (3147 articles) ===

--- Осъдиха шофьора, блъснал човек на АТВ в София ---
URL: https://www.vesti.bg/bulgaria/osydiha-shofiora-blysnal-chovek-na-atv-v-sofiia-6261280
С офийският градски съд (СГС) осъди на осем години лишаване от свобода Валентин Асенов, обвинен за катастрофата в столичния квартал „Христо Ботев“, при която на 14 юли 2025 г. един човек загина, каращ АТВ, а друг беше ранен, съобщиха от пресцентъра на съда.
Според съдебния състав подсъдимият е нарушил разпоредбите на Закона за движението по пътищата, като деянието е извършено по непредпазливост. Съдът постанови Валентин Асенов да изтърпи наложеното наказание при първоначален общ режим.
Оставиха в ареста Валентин Асенов, обвинен за катастрофата с АТВ в София
Съдът отне в полза на държавата лекия автомобил Бе Ем Ве, използван при извършването на престъплението.
Присъдата подлежи на обжалване и протест в 15-дневен срок от днес пред Софийския апелативен съд.
Припомняме, че през октомври 2025 г. Софийският апелативен 

In [83]:
mask = (df["source"] == "vesti") & df["full_text"].str.contains(
    "Още от автора", na=False
)
print(f"Vesti articles containing 'Още от автора': {mask.sum()}\n")

for _, row in df[mask].iterrows():
    print(f"--- {row['title']} ---")
    print(row["full_text"])
    print()

Vesti articles containing 'Още от автора': 13

--- <p>Зад кулисите на Мондиал 2026: Опасният съюз между Тръмп и шефа на ФИФА</p> ---
Б роени дни остават до началото на Световното първенство по футбол. Домакини на дългоочакваното събитие ще бъдат три страни – САЩ, Канада и Мексико. В надпреварата ще участват цели 48 отбора, което се случва за пръв път в историята. Турнирът ще започне на 12 юни в мексиканската столица Мексико Сити. Отвъд всички емоции и това, което предстои да се случи по терените, огромен интерес предизвикват някои политически въпроси, свързани с Мондиала. Сред тях са отношенията между две личности, които имат водещa роля в провеждането на първенството – държавния глава на САЩ Доналд Тръмп и президента на Международната футболна федерация ФИФА Джани Инфантино.
- Корупция, скандали и награда за мир
Още преди началото на Световното първенство, едно събитие от края на 2025 г. продължава да е обект на разпалени дебати и символизира необичайно близките отношения между Тръмп 

In [84]:
df = apply_to_source(
    df, lambda t: truncate_at_sentinel(t, ["Още от автора"]), source="vesti"
)

In [85]:
word_set = set(word for text in df["full_text"].dropna() for word in text.split())

# add manually some words that are not present in the corpus
word_set.update(
    ["Стегнатата", "Стрелбата", "Взискателният", "Синьо-зелен"]
)

In [86]:
vesti_df = df[df["source"] == "vesti"]
ambiguous_mask = vesti_df["full_text"].str.match(r"^[ВС] [а-я]", na=False)

for text in vesti_df[ambiguous_mask]["full_text"].head(10):
    parts = text.split()
    merged = parts[0] + parts[1]
    valid = merged in word_set
    print(f"'{parts[0]} {parts[1]}' → '{merged}' — {'✓' if valid else '✗'}")

'С евернокорейският' → 'Севернокорейският' — ✓
'В изуални' → 'Визуални' — ✓
'С и' → 'Си' — ✓
'В ицепрезидентът' → 'Вицепрезидентът' — ✓
'С инът' → 'Синът' — ✓
'С ветовното' → 'Световното' — ✓
'В ъзможно' → 'Възможно' — ✓
'С офийският' → 'Софийският' — ✓
'В столицата' → 'Встолицата' — ✗
'С пасители' → 'Спасители' — ✓


In [87]:
df = apply_to_source(df, lambda t: fix_split_first_letter(t, word_set), source="vesti")

In [88]:
mask = df[df["source"] == "vesti"]["full_text"].str.match(r"^[А-Я] [а-я]", na=False)
print(f"Remaining split first letters: {mask.sum()}\n")

for text in df[df["source"] == "vesti"][mask]["full_text"]:
    print(text[:80])

Remaining split first letters: 89

В столицата на Нова Зеландия Уелингтън бяха евакуирани стотици жители на крайбре
В рамките на тазгодишното издание на кампанията #PlantHealth4Life Българската аг
С настъпването на пролетта и началото на лятото Спасителният център за диви живо
В следващите дни Европейският съюз може да въведе нови санкции срещу израелски з
В събота прокурорите започнаха разследване срещу 95-годишен милиардер и почетен 
С мразяващо обаждане на спешния телефон 911 вдигна на крак полицейското управлен
В сряда ще бъде предимно слънчево, но в следобедните часове, главно над Източна 
В аналите на историята има малко гримоари (книги с магии), които да вдъхват толк
С документ под надслов "От фрагментация към мащаб - ускоряване на конкурентоспос
С документ под надслов "От фрагментация към мащаб - ускоряване на конкурентоспос
В най-късметлийския ден за вашата зодия през следващата седмица (от 8 до 14 юни 
В свят, в който животът ни преминава пред екрана, а дигиталната зависимост

### Vesti — Cleaning Summary

- Dropped social media attribution lines (e.g. `— Reuters (@Reuters) May 8, 2026`)
- Dropped lines containing only Latin characters (English sentences, social handles)
- Truncated text at `Още от автора`
- Merged split drop-cap first letters (e.g. `В ластите` → `Властите`), using corpus word lookup to avoid incorrectly joining valid single-letter prepositions


## 4.b. Monitor

In [89]:
print_source_samples(df, "monitor", n=3, random_state=42)

=== MONITOR (3404 articles) ===

--- Путин отряза Зеленски! ---
URL: https://telegraph.bg/svetat/novini/putin-otriaza-zelenski-otkaza-sreshta-moskva-shte-goni-voennite-si-celi-493633
Руският президент Владимир Путин отряза украинския си колега Володимир Зеленски.
Той заяви, че не вижда възможност за среща в близко бъдеще и подчерта, че конфликтът ще продължи, докато Москва не постигне поставените си военни цели.
Зеленски по-рано отправи публичен призив за директна среща между двамата лидери с цел ускоряване на дипломатическо решение и прекратяване на войната. Путин отхвърли предложението, като посочи, че подобна среща би имала смисъл единствено след изготвяне на конкретни договорености между експертни екипи и постигане на предварителни споразумения.
По думите му военните действия ще приключат единствено когато Русия реализира целите на своята операция. Той подчерта, че засега няма основания за политически диалог на най-високо ниво.
От своя страна Зеленски определи отказа като доказател

In [90]:
df = apply_to_source(df, lambda t: fix_split_first_letter(t, word_set), source="monitor")

In [91]:
mask = df[df["source"] == "monitor"]["full_text"].str.match(r"^[А-Я] [а-я]", na=False)
print(f"Remaining split first letters: {mask.sum()}\n")

for text in df[df["source"] == "monitor"][mask]["full_text"]:
    print(text[:80])

Remaining split first letters: 85

В историческия дворец Уестминстър, лорд Евгени Минчев стана първият българин удо
В японския град Уцуномия мечка, която броди по улиците от три дни, наложи затвар
В понеделник във вестник „Мач Телеграф“
* Последна част от поредицата ни за Монд
С началото на активния летен туристически сезон във Варна се активираха джебчиит
В неделя на пазара има само един спортен вестник - "Мач Телеграф"
* Най-важното 
В лична година 4-ка ще се захванете с доста работа или личната ви отговорност ще
В събота във вестник "Мач Телеграф"
* Приложение за Мондиал 2026 - група К - отб
В петък във вестник "Мач Телеграф"
* Приложение за Мондиал 2026 - група J - отбо
В нощта между четвъртък и петък Земята ще бъде ударена от две мощни изригвания н
С най-голяма надценка от борсата до трапезата са родните зеленчуци. Това показа 
С мразяващо предупреждение за бъдещето на човечеството направи професорът, нарич
В публикувано днес интервю президентът на САЩ Доналд Тръмп потвърди по-ран

In [92]:
word_set.update(
    [
        "Стегнатата",
        "Стрелбата",
        "Взискателният",
        "Синьо-зелен",
        "Суперзвездите",
        "Смразяваща",
        "Скандална",
    ]
)

In [93]:
mask = (df["source"] == "monitor") & df["full_text"].str.contains(
    "АБОНИРАЙТЕ СЕ", case=False, na=False
)
print(f"Monitor articles containing 'АБОНИРАЙТЕ СЕ': {mask.sum()}\n")

for _, row in df[mask].head(3).iterrows():
    print(f"--- {row['title']} ---")
    print(row["full_text"][20:])
    print()

Monitor articles containing 'АБОНИРАЙТЕ СЕ': 565

--- ИДВА ЛИ ЛЯТОТО? Чака ни жега с температури до 32 градуса! ---
ъде предимно слънчево. Около и след обяд главно над планините и Североизточна България ще се развива купеста и купесто-дъждовна облачност и на места там ще превали и прегърми. Ще духа слаб вятър от изток-североизток. Максималните температури ще бъдат предимно между 27° и 32°, по Черноморието – между 22° и 26°, а в София – около 28°.
В планините преди обяд ще бъде предимно слънчево. Около и след обяд ще се развива купеста и купесто-дъждовна облачност и на места, главно в Рило-Родопския масив и западните и централните дялове на Стара планина, ще има краткотрайни валежи и гръмотевици. Ще духа слаб до умерен вятър от изток-североизток, по най-високите върхове – от северозапад. Максималната температура на височина 1200 метра ще бъде около 22°, на 2000 метра – около 14°.
По Черноморието ще бъде предимно слънчево. В сутрешните часове на места видимостта ще е намалена. След обяд 

In [94]:
df = apply_to_source(
    df,
    lambda t: drop_lines(t, lambda line: "абонирайте се" in line.lower()),
    source="monitor",
)

In [95]:
print_source_samples(df, "monitor", n=3)

=== MONITOR (3404 articles) ===

--- ВАТИКАНЪТ СРЕЩУ МАШИНИТЕ: Папа Лъв XIV призова за „разоръжаване“ на изкуствения интелект ---
URL: https://telegraph.bg/svetat/novini/vatikanyt-sreshtu-mashinite-papa-lyv-xiv-prizova-za-razoryzhavane-na-izkustveniia-intelekt.-svetiiat-otec-izdade-pyrvata-si-enciklika-v-koiato-obiavi-algoritmite-za-moralno-opasni-492492
Ватиканът обяви фронтална война на безконтролното развитие на технологиите. В публикуваната си днес първа енциклика папа Лъв XIV отправи драматичен апел за незабавно „разоръжаване“ на изкуствения интелект (ИИ), за да се предотврати надмощието на машините над човечеството. Главата на Римокатолическата църква предупреди, че дигиталната революция не е неутрална и вече копае дълбока бездна между приобщените и изолираните членове на обществото, предаде Франс прес.
Машините не са морално чисти
Папа Лъв XIV с предупреждение за изкуствения интелект!
Историческият документ от 130 страници, озаглавен „Magnifica Humanitas“ („Великолепното човечес

### Monitor — Cleaning Summary

- Merged split drop-cap first letters
- Dropped lines containing `АБОНИРАЙТЕ СЕ` (subscription prompt injected into article text)


## 4.c. STANDARTNEWS

In [5]:
print_source_samples(df, "standartnews", n=3, random_state=42)

=== STANDARTNEWS (3921 articles) ===

--- Важно за бъдещите първолаци! ---
URL: https://www.standartnews.com/balgariya-obrazovanie/vazhno-za-badeshtite-parvolaci-635622.html
Приемът на ученици в първи клас в София продължава по график, въпреки решението на Административния съд - София-град, с което бяха отменени част от последните промени в наредбата за прием в общинските училища.
Първото класиране на 2 юни остава без промяна, а от Столичната община уточняват, че съдебното решение няма да повлияе на текущата кампания.
За предстоящите класирания за учебната 2026/2027 година родителите трябва да посочат дали детето ще кандидатства с постоянен или с настоящ адрес, съобщава dariknews.bg.
Изборът се прави чрез платформата ИСОДЗ, ПГУ и първи клас, като в меню „Критерии“ е добавен специален жълт бутон за заявяване на адреса.
Промяната на избрания адрес може да бъде направена до заключването на системата в 18:00 часа на 1 юни 2026 г.
След участието в първото класиране избраният адрес се запазв

In [97]:
df = apply_to_source(
    df,
    lambda t: truncate_at_sentinel(
        t, ["Последвайте ни в Google News Showcase за важните новини"]
    ),
    source="standartnews",
)

### Standartnews — Cleaning Summary

- Dropped lines starting with `Последвайте ни в Google News Showcase за важните новини` till end of the article


## 4.d. SEGABG 

In [98]:
print_source_samples(df, "segabg", n=3, random_state=42)

=== SEGABG (1567 articles) ===

--- Новият унгарски премиер намали заплатата си наполовина ---
URL: https://www.segabg.com/hot/category-foreign-country/noviyat-ungarski-premier-namali-zaplatata-si-napolovina
Новият унгарски премиер намали собствената си заплата наполовина и съкрати възнагражденията на министрите си, за да си върне доверието на обществото, съобщава "Франкфуртер Рундшау", предаде БТА.
За да "даде пример" след години на предполагаема корупция в правителството. Петър Мадяр обяви, че ще намали годишното си възнаграждение до 45,6 милиона форинта (128 040 евро), което е по-малко от половината от 93,6 милиона (263 424 евро), колкото получаваше предшественикът му Виктор Орбан.
Мадяр обясни, че намаляването на заплатите ще важи и за ръководителите на държавни предприятия, както и за членовете на надзорни и консултативни органи на държавни фирми. Според неговите изчисления мерките биха могли да спестят около 139,91 млн. евро през текущия законодателен мандат. Освен това министрит

In [99]:
df = apply_to_source(df, lambda t: drop_lines(t, is_noise_line), source="segabg")

In [100]:
print_source_samples(df, "segabg", n=3)

=== SEGABG (1567 articles) ===

--- Бизнесът зове властта да се откаже от налагане на &quot;справедливи цени&quot; ---
URL: https://www.segabg.com/hot/category-economy/biznesut-zove-vlastta-da-se-otkazhe-nalagane-na-spravedlivi-ceni
Да отпадне понятието „справедлива цена“ като законова категория, а забраната на "икономически необосновано“ увеличение на цените да се преосмисли, така че да не бъде приложима към практически всяко ценово движение.
За това настояха работодателските организации на днешното заседание на съвета за тристранно сътрудничество, на което бяха обсъдени промените в Закона за защита на потребителите и Закона за защита на конкуренцията, с които новата власт иска да овладее ръста на цените. Обсъждането в тристранката се случва, след като промените вече са приети в НС на първо четене.
Междувременно премиерът Румен Радев реши да се срещне в късния следобед с представители на търговските вериги, за да обсъди същите промени.
Двата законопроекта се разглеждат заедно, защото 

In [101]:
mask = (df["source"] == "segabg") & df["title"].str.startswith(
    "Спортът по телевизията", na=False
)
print(f"Segabg articles starting with 'Спортът по телевизията': {mask.sum()}")

Segabg articles starting with 'Спортът по телевизията': 37


In [102]:
def drop_by_field(df, pattern, field="full_text", source=None, match="startswith"):
    patterns = [pattern] if isinstance(pattern, str) else pattern
    combined = "|".join(patterns)

    if match == "startswith":
        mask = df[field].str.startswith(tuple(patterns), na=False)
    else:
        mask = df[field].str.contains(combined, case=False, na=False)

    if source:
        mask = mask & (df["source"] == source)
    print(f"Dropping {mask.sum()} articles where {field} {match} {patterns}")
    df = df[~mask].reset_index(drop=True)
    print(f"Remaining articles: {len(df)}")
    return df

In [103]:
df = drop_by_field(df, ["Спортът по телевизията"], field="title", source="segabg")

Dropping 37 articles where title startswith ['Спортът по телевизията']
Remaining articles: 51219


### Segabg — Cleaning Summary

- Dropped social media attribution lines 
- Dropped lines starting with emojis (embedded tweet/social media posts) - applied directly on whole dataset, not only on `segabg` source
- Dropped boilerplate TV schedule articles (`Спортът по телевизията`)


## 4.e. BTA

In [104]:
print_source_samples(df, "bta", n=3, random_state=42)

=== BTA (4633 articles) ===

--- Спадът на раждаемостта насочва „Нестле“ към пазара на храни за домашни любимци ---
URL: https://www.bta.bg/bg/news/economy/1139087-spadat-na-razhdaemostta-nasochva-nestle-kam-pazara-na-hrani-za-domashni-lyubim
site.btaСпадът на раждаемостта насочва „Нестле“ към пазара на храни за домашни любимци
Швейцарският концерн „Нестле“ (Nestlé) залага все по-силно на бизнеса с храни за домашни любимци на фона на спадащата раждаемост в световен мащаб, предаде Ройтерс.
Главният изпълнителен директор на компанията Филип Навратил заяви по време на инвеститорска конференция на „Дойче Банк“ (Deutsche Bank), че тенденцията към раждането на по-малко бебета и гледането на повече домашни любимци създава значителен потенциал за растеж.
По думите му собствениците на домашни любимци са все по-склонни да увеличават разходите за тях, тъй като животните все по-често се възприемат като пълноценни членове на семейството.
Компанията произвежда храни за домашни любимци под марките „П

In [105]:
def strip_prefix(text: str, prefix: str) -> str:
    if not isinstance(text, str):
        return text
    if text.startswith(prefix):
        return text[len(prefix) :].strip()
    return text

In [106]:
df = apply_to_source(df, lambda t: strip_prefix(t, "site.bta"), source="bta")
df = apply_to_source(df, lambda t: truncate_at_sentinel(t, ["/"]), source="bta")

In [107]:
print_source_samples(df, "bta", n=3)

=== BTA (4633 articles) ===

--- Нов общински съветник положи клетва в Общински съвет – Габрово ---
URL: https://www.bta.bg/bg/news/bulgaria/1124615-nov-obshtinski-savetnik-polozhi-kletva-v-obshtinski-savet-gabrovo
Нов общински съветник положи клетва в Общински съвет – Габрово
Нов общински съветник положи клетва в Общински съвет – Габрово. Лена Енева встъпи в състава му на мястото на Николай Косев, който бе избран за народен представител.
В началото на заседанието общинският съветник Иван Иванов обърна внимание на събитията от последните дни, свързани с напрежение в Габрово след масово сбиване. Председателят Климент Кунев посочи, че при изясняване на всички факти по случая ще бъдат поканени компетентни лица и ще бъде организирано изслушване в общинския съвет.
Общинският съвет одобри предложението Лена Енева да бъде член на две комисии.
Съветниците приеха и предложението за избор на Детелин Цветков за управител на „Общински пътнически транспорт“ ЕООД.
Дискусия предизвика точката за подп

### BTA — Cleaning Summary

- Stripped `site.bta` prefix appearing at the start of some articles
- Dropped lines starting with `/` (navigation artifacts from the site structure)


## 4.f. BANKER

In [108]:
print_source_samples(df, "banker", n=3, random_state=42)

=== BANKER (997 articles) ===

--- Най-доброто време за ядене на орехи за максимално усвояване на омега-3 ---
URL: https://banker.bg/2026/05/21/vreme-za-yadene-na-orehi-na-omega-3/
Орехите може да са малки, но са изключително хранителни. Те са удобна храна за съхранение у дома и съдържат фибри, протеини, магнезий и полезни мазнини. Освен това орехите са един от най-добрите растителни източници на омега-3 благодарение на съдържащата се в тях алфа-линоленова киселина (ALA).
Може би знаете, че някои хранителни вещества се усвояват по-добре, когато се приемат с определени храни или по определено време. Затова, ако ядете орехи заради омега-3 ползите им, вероятно се питате дали времето има значение.
Най-доброто време за ядене на орехи
Диетолозите са единодушни: няма конкретно най-добро време с цел максимално усвояване на омега-3. Най-важното е те да присъстват редовно в балансираното хранене.
„Добрата новина е, че независимо кога се консумират, орехите носят ползи за здравето“, казва Рейчъл 

## 4.g. ECONOMIC

In [109]:
print_source_samples(df, "economic", n=3, random_state=42)

=== ECONOMIC (466 articles) ===

--- Здравеопазването като геополитическо оръжие: Европа създава медицински шампиони, но ги губи ---
URL: https://www.economic.bg/bg/a/view/zdraveopazvaneto-kato-geopolitichesko-oryjie-evropa-syzdava-medicinski-shampioni-no-gi-gubi
Здравеопазването като геополитическо оръжие: Европа създава медицински шампиони, но ги губи
Изпълнителният директор на Astellas Pharma за България, Румъния, Унгария и Гърция доц. д-р Светослав Ценов предупреждава, че Европа се нуждае от спешна промяна в регулациите, за да задържи технологиите на бъдещето
Въпреки че Европа остава дом на някои от най-добрите медицински университети и изследователски центрове в света, континентът хронично изостава в една критична фаза – комерсиализацията и мащабирането на своите открития. Този отрезвяващ извод бе в центъра на дискусиите по време на ключовия панел на Green Transition Forum 6.0, посветен на европейската конкурентоспособност.
В дебат, споделяйки сцената с Енрико Лета, автор на знако

In [110]:
df.groupby([df["published_at_dt"].dt.date, "source"]).size().unstack(fill_value=0)

source,24chasa,actualno,banker,bta,economic,fakti,monitor,nova,segabg,standartnews,vesti
published_at_dt,,,,,,,,,,,
2026-04-19,0,0,0,0,0,0,0,5,0,0,0
2026-04-20,115,66,0,0,0,36,0,104,0,0,0
2026-04-21,304,211,0,0,0,183,0,118,0,0,0
2026-04-22,307,212,0,0,0,176,0,117,0,0,0
2026-04-23,315,187,0,0,0,183,0,109,0,0,0
2026-04-24,303,196,0,0,0,182,0,112,3,0,0
2026-04-25,218,80,0,0,0,169,0,75,1,0,0
2026-04-26,162,101,0,0,0,150,0,84,0,0,0
2026-04-27,302,193,0,0,0,155,0,106,3,0,0


### Economic & Banker — Cleaning Summary

No source-specific cleaning required — article texts are well-structured with no significant boilerplate, encoding issues, or extraction artifacts.

**Note:** Both sources publish a relatively small number of articles per day compared to the larger sources. The same applies to `segabg`. Whether to keep them depends on the downstream tasks:
- For **event clustering and temporal summarization**, small sources are still valuable — they contribute additional perspectives on the same events covered by larger sources.
- The risk is **underrepresentation** — if an event is covered only by a small source, the cluster may be too thin for meaningful summarization.

**Decision:** Keep all three sources for now and revisit after clustering — if clusters from these sources are consistently too small to summarize, they can be excluded at that stage.


## 4.h. Actualno

In [111]:
print_source_samples(df, "actualno", n=3, random_state=42)

=== ACTUALNO (7338 articles) ===

--- В Разград се хванаха за главите: Ненужен в Лудогорец не иска да си ходи и извива ръцете на шефовете ---
URL: https://www.actualno.com/bgfootball/v-razgrad-se-hvanaha-za-glavite-nenujen-v-ludogorec-ne-iska-da-si-hodi-i-izviva-rycete-na-shefovete-news_2601730.html
Шефовете на Лудогорец се хванаха за главите и се чудят какво да правят с един от ненужните футболисти в състава на „орлите“. Става въпрос за централния бранител на разградчани – Оливие Вердон. Както е известно, националът на Бенин не попада в плановете на клуба за следващия сезон и поради тази причина ръководството търси начин да се раздели с него. На този етап обаче „удрят греда“.
Вердон няма никакво намерение да „скъса“ договора си по взаимно съгласие
Според информация на колегите от „Тема спорт“ ръководството на Лудогорец и Вердон са стартирали преговори за разтрогване на договора му, който е до лятото на 2027 година. Централният бранител обаче е категоричен, че не възнамерява да „скъса“

In [112]:
df = apply_to_source(
    df,
    lambda t: drop_lines(
        t, lambda line: line.strip().startswith(("Снимка:", "Източник:"))
    ),
    source="actualno",
)

In [113]:
df = apply_to_source(df, lambda t: drop_lines(t, is_noise_line), source="actualno")

### Actualno — Cleaning Summary

- Dropped lines starting with `Снимка:` and `Източник:` (photo and source attribution labels)
- Dropped lines containing only Latin characters (embedded tweets and foreign-language content)

**Pending decision:** Lines starting with `Още:` are links to related articles. These are not part of the article content but are kept for now — will revisit after evaluating their impact on embeddings.


## 4.i. NOVA

In [114]:
print_source_samples(df, "nova", n=3, random_state=42)

=== NOVA (5139 articles) ===

--- Морето взе първа жертва в навечерието на откриването на летния сезон ---
URL: https://nova.bg/news/view/2026/06/01/539098/%D0%BC%D0%BE%D1%80%D0%B5%D1%82%D0%BE-%D0%B2%D0%B7%D0%B5-%D0%BF%D1%8A%D1%80%D0%B2%D0%B0-%D0%B6%D0%B5%D1%80%D1%82%D0%B2%D0%B0-%D0%B2-%D0%BD%D0%B0%D0%B2%D0%B5%D1%87%D0%B5%D1%80%D0%B8%D0%B5%D1%82%D0%BE-%D0%BD%D0%B0-%D0%BE%D1%82%D0%BA%D1%80%D0%B8%D0%B2%D0%B0%D0%BD%D0%B5%D1%82%D0%BE-%D0%BD%D0%B0-%D0%BB%D0%B5%D1%82%D0%BD%D0%B8%D1%8F-%D1%81%D0%B5%D0%B7%D0%BE%D0%BD/
Снимка: iStock
Жена се удави на плажа в Несебър
Жена се удави на плажа "Олимпийски надежди" в Несебър. Сигналът за забелязана в морето давеща се жена е подаден в неделя около 20:30 ч.
Тялото е извадено от водата, а самоличността на загиналата все още не е установена.
При извършения оглед не са открити видими следи от насилие по тялото на жертвата. На брега са намерени плажната ѝ чанта с летни дрехи и джапанки. Трупът е откаран за аутопсия в Бургас.
Последвайте ни

--- Два тира ка

In [115]:
text = "Новините на NOVA Още от Nova Новините на Нова – емисии Начало Новините на NOVA Новините на NOVA (30.04.2026 - късна) Начало Още от Nova Новините на NOVA (30.04.2026 - късна) Начало Новините на Нова – емисии Новините на NOVA (30.04.2026 - късна) Новините на NOVA (30.04.2026 - късна) 30 април 2026 23:02 Новините на NOVA NEWS (30.04.2026 - 20:00) Новините на NOVA (30.04.2026 - централна) Новините на NOVA (30.04.2026 - следобедна) Обеден информационен блок на NOVA NEWS (30.04.2026) Новините на NOVA (30.04.2026 - обедна) Новините на NOVA (30.04.2026 - 9.00) Гледайте цялата емисия Редактор: Ивета Костадинова Последвайте ни NewsLetter Google News Youtube Viber TikTok Instagram Facebook Още от Още от Nova Новините на NOVA NEWS (30.04.2026 - 20:00) Новините на NOVA (30.04.2026 - централна) „Пресечна точка”: За 52-рото Народно събрание и очакванията на президента Илияна Йотова Новините на NOVA (30.04.2026 - следобедна) Обеден информационен блок на NOVA NEWS (30.04.2026) Новините на NOVA (30.04.2026 - обедна) Новините на NOVA (30.04.2026 - 9.00) Новините на NOVA (30.04.2026 - 8.00) Новините на NOVA (30.04.2026 - 7.00) Новините на NOVA (30.04.2026 - 6.00) Прогноза за времето (30.04.2026 - сутрешна) Новините на NOVA (29.04.2026 - късна) Новините на NOVA NEWS (29.04.2026 - 20:00) Новините на NOVA (29.04.2026 - централна) „Пресечна точка”: За закриването на „Магазин за хората” и българското здравеопазване Новините на NOVA (29.04.2026 - следобедна) Обеден информационен блок на NOVA NEWS (29.04.2026) Новините на NOVA (29.04.2026 - обедна) Новините на NOVA (29.04.2026 - 9.00) Новините на NOVA (29.04.2026 - 8.00) Новините на NOVA (29.04.2026 - 7.00) Новините на NOVA (29.04.2026 - 6.00) Новините на NOVA (28.04.2026 - късна) Новините на NOVA (28.04.2026 - централна) „Пресечна точка”: За съдебната система и за заплатите в държавния сектор Новините на NOVA (28.04.2026 - следобедна) Обеден информационен блок на NOVA NEWS (28.04.2026) Новините на NOVA (28.04.2026 - обедна) Новините на NOVA (28.04.2026 - 9.00) Новините на NOVA (28.04.2026 - 8.00) Новините на NOVA (28.04.2026 - 7.00) Новините на NOVA (28.04.2026 - 6.00) Новините на NOVA (27.04.2026 - късна) Новините на NOVA NEWS (27.04.2026 - 20:00) Новините на NOVA (27.04.2026 - централна) 'Пресечна точка': За състава на 52-рото Народно събрание и ръста на цените у нас Новините на NOVA (27.04.2026 - следобедна) Обеден информационен блок на NOVA NEWS (27.04.2026) Новините на NOVA (27.04.2026 - обедна) Новините на NOVA (27.04.2026 - 9.00) Новините на NOVA (27.04.2026 - 8.00) Новините на NOVA (27.04.2026 - 7.00) Новините на NOVA (27.04.2026 - 6.00) Новините на NOVA NEWS (26.04.2026 - 20:00) Новините на NOVA (26.04.2026 - централна) Новините на NOVA (26.04.2026 - обедна) Прогноза за времето (26.04.2026 - обедна) Спортни новини (26.04.2026 - обедна) ОЩЕ ОТ КАТЕГОРИЯТА"


def print_wrapped(text: str, words_per_line: int = 100) -> str:
    words = text.split()
    lines = []
    for i in range(0, len(words), words_per_line):
        lines.append(" ".join(words[i : i + words_per_line]))
    return "\n".join(lines)

In [116]:
print(print_wrapped(text, words_per_line=20))

Новините на NOVA Още от Nova Новините на Нова – емисии Начало Новините на NOVA Новините на NOVA (30.04.2026 -
късна) Начало Още от Nova Новините на NOVA (30.04.2026 - късна) Начало Новините на Нова – емисии Новините на NOVA
(30.04.2026 - късна) Новините на NOVA (30.04.2026 - късна) 30 април 2026 23:02 Новините на NOVA NEWS (30.04.2026 - 20:00)
Новините на NOVA (30.04.2026 - централна) Новините на NOVA (30.04.2026 - следобедна) Обеден информационен блок на NOVA NEWS (30.04.2026) Новините
на NOVA (30.04.2026 - обедна) Новините на NOVA (30.04.2026 - 9.00) Гледайте цялата емисия Редактор: Ивета Костадинова Последвайте ни NewsLetter
Google News Youtube Viber TikTok Instagram Facebook Още от Още от Nova Новините на NOVA NEWS (30.04.2026 - 20:00) Новините
на NOVA (30.04.2026 - централна) „Пресечна точка”: За 52-рото Народно събрание и очакванията на президента Илияна Йотова Новините на NOVA
(30.04.2026 - следобедна) Обеден информационен блок на NOVA NEWS (30.04.2026) Новините на NOVA (30.04.2

In [117]:
def check_field_start(df, pattern, field="full_text", source=None):
    mask = df[field].str.startswith(pattern, na=False)
    if source:
        mask = mask & (df["source"] == source)
    print(f"Articles where {field} starts with '{pattern}': {mask.sum()}")
    return df[mask].head()

In [118]:
check_field_start(df, pattern="Новините на NOVA", field="title", source="nova")

Articles where title starts with 'Новините на NOVA': 347


,source,title,url,published_at,full_text,fetched_at,published_at_dt,word_count
216,nova,Новините на NOVA (09.06.2026 - обедна),https://nova.bg/news/view/2026/06/09/540043/%D...,2026-06-09T09:31:39+00:00,Новините на NOVA Още от Nova Новините на Нова ...,2026-06-09T11:19:14.061560+00:00,2026-06-09 09:31:39+00:00,445
536,nova,Новините на NOVA (09.06.2026 - 7.00),https://nova.bg/news/view/2026/06/09/540000/%D...,2026-06-09T04:10:20+00:00,Новините на NOVA Още от Nova Новините на Нова ...,2026-06-09T07:53:22.708802+00:00,2026-06-09 04:10:20+00:00,446
545,nova,Новините на NOVA (09.06.2026 - 8.00),https://nova.bg/news/view/2026/06/09/539999/%D...,2026-06-09T05:12:01+00:00,Новините на NOVA Още от Nova Новините на Нова ...,2026-06-09T07:53:22.708720+00:00,2026-06-09 05:12:01+00:00,446
553,nova,Новините на NOVA (09.06.2026 - 9.00),https://nova.bg/news/view/2026/06/09/539997/%D...,2026-06-09T06:10:50+00:00,Новините на NOVA Още от Nova Новините на Нова ...,2026-06-09T07:53:22.708645+00:00,2026-06-09 06:10:50+00:00,446
727,nova,Новините на NOVA (09.06.2026 - 6.00),https://nova.bg/news/view/2026/06/09/539998/%D...,2026-06-09T03:15:28+00:00,Новините на NOVA Още от Nova Новините на Нова ...,2026-06-09T03:42:30.901240+00:00,2026-06-09 03:15:28+00:00,455


In [119]:
df = drop_by_field(df, ["Новините на NOVA"], field="title", source="nova")

Dropping 347 articles where title startswith ['Новините на NOVA']
Remaining articles: 50872


In [120]:
check_field_start(df, pattern="Прогноза за времето", field="title", source="nova")

Articles where title starts with 'Прогноза за времето': 90


,source,title,url,published_at,full_text,fetched_at,published_at_dt,word_count
217,nova,Прогноза за времето (08.06.2026 - обедна),https://nova.bg/news/view/2026/06/09/540045/%D...,2026-06-09T09:50:13+00:00,Новините на NOVA България Времето Начало Новин...,2026-06-09T11:19:14.061541+00:00,2026-06-09 09:50:13+00:00,667
720,nova,Прогноза за времето (09.06.2026 - сутрешна),https://nova.bg/news/view/2026/06/09/540001/%D...,2026-06-09T03:07:42+00:00,Новините на NOVA Още от Nova Времето Начало Но...,2026-06-09T03:42:30.901272+00:00,2026-06-09 03:07:42+00:00,459
1619,nova,Прогноза за времето (08.06.2026 - обедна),https://nova.bg/news/view/2026/06/08/539917/%D...,2026-06-08T09:45:58+00:00,Новините на NOVA България Времето Начало Новин...,2026-06-08T11:01:54.615019+00:00,2026-06-08 09:45:58+00:00,622
1840,nova,Прогноза за времето (08.06.2026 - сутрешна),https://nova.bg/news/view/2026/06/08/539882/%D...,2026-06-08T03:44:21+00:00,Новините на NOVA България Времето Начало Новин...,2026-06-08T04:14:12.343973+00:00,2026-06-08 03:44:21+00:00,648
2360,nova,Прогноза за времето (07.06.2026 - обедна),https://nova.bg/news/view/2026/06/07/539814/%D...,2026-06-07T09:45:16+00:00,Новините на NOVA Още от Nova Времето Начало Но...,2026-06-07T11:50:32.142609+00:00,2026-06-07 09:45:16+00:00,469


In [121]:
df = drop_by_field(df, "Прогноза за времето", field="title", source="nova")

Dropping 90 articles where title startswith ['Прогноза за времето']
Remaining articles: 50782


In [122]:
df = drop_by_field(df, "Новините на NOVA", field="full_text", source="nova")

Dropping 325 articles where full_text startswith ['Новините на NOVA']
Remaining articles: 50457


In [123]:
df = apply_to_source(
    df, lambda t: truncate_at_sentinel(t, ["Последвайте ни"]), source="nova"
)

In [124]:
df = apply_to_source(
    df,
    lambda t: drop_lines(
        t, lambda line: line.strip().startswith(("Снимка:", "Снимки:"))
    ),
    source="nova",
)

In [125]:
df = apply_to_source(df, lambda t: drop_lines(t, is_noise_line), source="nova")

In [126]:
phrases = [
    "Повече гледайте във видеото",
    "Редактор:",
    "Цялото предаване гледайте във видеото",
]

for phrase in phrases:
    mask = (df["source"] == "nova") & df["full_text"].str.contains(phrase, na=False)
    print(f"'{phrase}': {mask.sum()} articles")

'Повече гледайте във видеото': 396 articles
'Редактор:': 3094 articles
'Цялото предаване гледайте във видеото': 28 articles


In [127]:
df = apply_to_source(
    df,
    lambda t: drop_lines(
        t,
        lambda line: line.strip().startswith(
            (
                "Повече гледайте във видеото",
                "Редактор:",
                "Цялото предаване гледайте във видеото",
            )
        ),
    ),
    source="nova",
)

### Nova — Cleaning Summary

- Dropped articles with `title` or `full_text` starting with `Новините на NOVA` and `Прогноза за времето`
- Truncated text at `Последвайте ни`
- Dropped lines starting with `Снимка:` and `Снимки:` (photo captions)
- Dropped lines starting with `Редактор:` (editor attribution)
- Dropped lines containing only Latin characters (foreign text, social media)
- Dropped social media attribution lines (e.g. `— Reuters (@Reuters) May 8, 2026`)
- Dropped emoji-heavy and `@handle` lines (embedded tweets and TikTok posts)
- Dropped boilerplate lines: `Повече гледайте във видеото`, `Цялото предаване гледайте във видеото`


## 4.j. 24CHASA

In [128]:
print_source_samples(df, "24chasa", n=3, random_state=42)

=== 24CHASA (13052 articles) ===

--- Украинка неуспешно опита да продаде 6-годишния си син за 25 000 долара ---
URL: https://www.24chasa.bg/mezhdunarodni/article/22722534
Украинка се опита да продаде 6-годишния си син за 25 000 долара.
38-годишна жена е била задържана, съобщиха от украинската прокуратура. На жената е повдигнато обвинение за трафик на непълнолетни.
По данни от разследването, след като се развела, тя заживяла с най-малкия си син, докато другите ѝ две деца - на 13 и 16 години - останали при баща си. Жената, която вече е осъждана за кражби, решила да се освободи от грижите за детето, като го продаде.
През март 2026 г. тя намерила „купувач" и се договорила за предаването на момчето срещу парична сума. Мъжът, на когото било предложено да купи детето, сигнализирал органите на реда.
Предаването на детето се състояло на 17 април 2026 г. в град Луцк в рамките на специализирана полицейска операция. Веднага след получаването на парите жената е била задържана.
Съдът вече е определ

In [129]:
phrase = "Разберете големите новини от последното денонощие накуп Запиши се"

mask = (df["source"] == "24chasa") & df["full_text"].str.contains(phrase, na=False)
print(f"Articles containing phrase: {mask.sum()}\n")

for _, row in df[mask].head().iterrows():
    normalized = re.sub(r"\s+", " ", row["full_text"])
    count = normalized.count(phrase)
    print(f"--- {row['title']} (appears {count}x) ---")
    print_by_words(row["full_text"])
    print()

multi = (
    df[mask]["full_text"]
    .apply(lambda t: re.sub(r"\s+", " ", t).count(phrase) > 1)
    .sum()
)
print(f"Total articles with phrase appearing more than once: {multi}")

Articles containing phrase: 675

--- Цената на петрола на ОПЕК вече е под 97 долара за барел (appears 1x) ---
Цената на петрола на ОПЕК падна под 97 долара за барел Снимка: СНИМКА: Pixabay (илюстративна) "24 часа" в мейла ти.
Разберете големите новини от последното денонощие накуп Запиши се Цената на петрола на Организацията на страните износителки на петрол (ОПЕК)
вчера се понижи за шеста поредна сесия - до 96,31 долара за барел, съобщи днес на страницата си петролният картел.
Петролът на ОПЕК се котира под 97 долара за барел за първи път от 6 март. Един барел петрол на
ОПЕК се продаваше за 97,74 долара в понеделник. На 27 февруари, преди началото на войната на САЩ и Израел срещу
Иран, петролът на ОПЕК се котираше за 70,07 долара за барел. За определяне на цената на петрола на ОПЕК се
използва т. нар. петролна кошница, включваща 12 сорта петрол за износ на страните членки на картела: Сахаран бленд или "Сахарска
смес" (Алжир), Джено (Конго), Зафиро (Екваториална Гвинея), Раби лек (Габо

**Finding:** The phrase `"Разберете големите новини от последното денонощие накуп Запиши се"` appears in a large portion of `24chasa` articles, always once and always at the beginning — it is a newsletter subscription prompt injected before the actual article content. The real text starts immediately after it.

**Decision:** Strip everything up to and including this phrase, keeping only the content that follows.


In [130]:
def strip_before_sentinel(text: str, sentinel: str) -> str:
    if not isinstance(text, str):
        return text
    idx = text.find(sentinel)
    if idx != -1:
        return text[idx + len(sentinel) :].strip()
    return text

In [131]:
PROMPT = "Разберете големите новини от последното денонощие накуп Запиши се"

mask = (df["source"] == "24chasa") & df["full_text"].str.contains(PROMPT, na=False)
print(f"Cleaning {mask.sum()} 24chasa articles with newsletter prompt")

df = apply_to_source(
    df,
    lambda t: (
        strip_before_sentinel(t, PROMPT)
        if isinstance(t, str) and PROMPT in re.sub(r"\s+", " ", t)
        else t
    ),
    source="24chasa",
)

Cleaning 675 24chasa articles with newsletter prompt


In [132]:
df = apply_to_source(df, lambda t: drop_lines(t, is_noise_line), source="24chasa")

In [133]:
def drop_phrase(text: str, phrases: list[str]) -> str:
    if not isinstance(text, str):
        return text
    for phrase in phrases:
        text = text.replace(phrase, "")
    return text.strip()

In [134]:
PHRASES = [
    '"24 часа" в мейла ти. Разберете големите новини от последното денонощие накуп',
]

mask = (df["source"] == "24chasa") & df["full_text"].apply(
    lambda t: any(p in t for p in PHRASES) if isinstance(t, str) else False
)
print(f"Records affected: {mask.sum()}\n")

for _, row in df[mask].iterrows():
    print(f"--- {row['title']} ---")
    print_by_words(row["full_text"])
    print()

Records affected: 252

--- Спорт по телевизията днес ---
"24 часа" в мейла ти. Разберете големите новини от последното денонощие накуп Последвайте ни в Google News Showcase Здравейте, благодарим
Ви, че четете "24 часа" редовно! Искаме да Ви поканим да се регистрирате безплатно в "24 часа", за да получите
пълен достъп до архива на сайта, възможност да персонализирате новините, които четете и абонамент за нашия нюзлетър. Благодарим ви! Благодарим
ви, че се включихте! Изтеглете новото приложение на "24 часа"

--- Само в "24 часа" на 9 юни: Най-пълният кандидат-студентски справочник ---
"24 часа" в мейла ти. Разберете големите новини от последното денонощие накуп Само в "24 часа" на 9 юни: Специални
страници с най-пълния справочник за кандидат-студентите Защо в България имаме най-високата инфлация в ЕС - анализ на икономиста Георги Ангелов
София опитва да си върне шест вкусни имота в Борисовата градина Вижте първите страници на "24 часа" от 9 юни
през годините до днес. Заглавията са интере

In [135]:
df = apply_to_source(df, lambda t: drop_phrase(t, PHRASES), source="24chasa")

In [136]:
phrase = "Последвайте ни в Google News Showcase"

mask = (df["source"] == "24chasa") & df["full_text"].str.contains(phrase, na=False)
print(f"Articles containing phrase: {mask.sum()}\n")

at_end = 0
not_at_end = 0

for _, row in df[mask].iterrows():
    text = row["full_text"]
    count = text.count(phrase)
    idx = text.find(phrase)
    total_len = len(text)
    position_pct = idx / total_len * 100
    remaining_words = len(text[idx + len(phrase) :].strip().split())

    if remaining_words <= 56:
        at_end += 1
    else:
        not_at_end += 1
        print(f"--- {row['title']} ---")
        print(
            f"Appears {count}x | Position: {position_pct:.0f}% into text | {remaining_words} words after"
        )
        print_by_words(row["full_text"])
        print()

print(f"\nAt end of text  : {at_end}")
print(f"Not at end      : {not_at_end}")

Articles containing phrase: 300

--- Ракетата-носител „Чанджън 12B Y1“ извърши успешно първия си изпитателен полет ---
Appears 1x | Position: 49% into text | 62 words after
На 1 юни в 16:40 часа пекинско време бе изстреляна успешно ракетата-носител „Чанджън 12B Y1", пише КМГ. Това бе първият
ѝ изпитателен полет. С нея бяха изведени в полярна орбита спътници от осмата група „Цянфън". На 1 юни в 16:40
часа пекинско време бе изстреляна успешно ракетата-носител „Чанджън 12B Y1", пише КМГ. Това бе първият ѝ изпитателен полет. С нея
бяха изведени в полярна орбита спътници от осмата група „Цянфън". Последвайте ни в Google News Showcase Това съдържание е подготвено
от журналисти на Китайската медийна група. Здравейте, благодарим Ви, че четете "24 часа" редовно! Искаме да Ви поканим да се
регистрирате безплатно в "24 часа", за да получите пълен достъп до архива на сайта, възможност да персонализирате новините, които четете
и абонамент за нашия нюзлетър. Благодарим ви! Благодарим ви, че се включ

In [137]:
df = apply_to_source(
    df,
    lambda t: truncate_at_sentinel(t, ["Последвайте ни в Google News Showcase"]),
    source="24chasa",
)

**Finding:** After thorough analysis, `Последвайте ни в Google News Showcase` followed by a newsletter/app subscription prompt was found to always appear at the end of the article text — it is an advertising/subscription block injected by the site, never part of the actual content.

**Decision:** Truncate everything from this phrase onwards.


In [138]:
df = drop_by_field(df, ["Спорт по телевизията днес"], field="title", source="24chasa")

Dropping 45 articles where title startswith ['Спорт по телевизията днес']
Remaining articles: 50412


In [139]:
short_mask = (df["source"] == "24chasa") & (df["word_count"] < 40)
print(f"24chasa articles under 30 words: {short_mask.sum()}\n")

for _, row in df[short_mask].iterrows():
    print(f"--- {row['title']} ({row['word_count']} words) ---")
    print(row["url"])
    print(row["full_text"])
    print()

24chasa articles under 30 words: 33

--- Ренета Колева става зам.-екоминистър (36 words) ---
https://www.24chasa.bg/bulgaria/article/23002946
Със заповед на министър-председателя Румен Радев на длъжността заместник-министър на околната среда и водите е назначена Ренета Георгиева Колева.
Това съобщиха от правителствената пресслужба. Другият зам.-екоминистър е Костадин Гешев, който бе назначен в края на май.

--- Верижна катастрофа затвори пътя София-Варна край Лясковец (38 words) ---
https://www.24chasa.bg/bulgaria/article/22994997
Временно е затворен главен път София -Варна, в района на разклона за Лясковец, заради катастрофа. Автомобилите се пренасочват през с. Козаревец. Това съобщи ОДМВР - Велико Търново.
По първоначални данни са се ударили три леки автомобила, има пострадали.

--- Южна Корея номинира жена за премиер за пръв път от 20 г. (38 words) ---
https://www.24chasa.bg/mezhdunarodni/article/22993851
Южнокорейският президент И Дже-мьон избра Хан Сонгсук, министър на малките и с

In [140]:
mask = df["title"].str.contains("изтегли приложението от тук", case=False, na=False)
print(f"Articles containing 'изтегли приложението от тук' in title: {mask.sum()}")

Articles containing 'изтегли приложението от тук' in title: 7


In [141]:
df = drop_by_field(df, ["изтегли приложението от тук"], field="title", match="contains")

Dropping 7 articles where title contains ['изтегли приложението от тук']
Remaining articles: 50405


In [142]:
df = apply_to_source(
    df,
    lambda t: drop_lines(t, lambda line: line.strip().lower().startswith("Снимка:")),
    source="24chasa",
)

In [143]:
mask = df["title"].str.startswith('"24 часа" на', na=False)
print(f'Articles containing "24 часа" на in title: {mask.sum()}\n')

for _, row in df[mask].iterrows():
    print(f"--- {row['title']} ---")
    print_by_words(row["full_text"])
    print()

Articles containing "24 часа" на in title: 2

--- "24 часа" на 16 май: Не, не е чудо, което те прави красив и млад – опасно е! ---
Само в "24 часа" на 16 май четете: Не, не е чудо, което те прави красив и млад – опасно
е! Тренд с пептиди превзе социалните мрежи, инфлуенсъри не просто рекламират неодобрените субстанции – продават ги през профилите си, регулации
няма - съботен очерк. Бангаранга е! Дайте й трофея вече! Дара излиза 12-а във финала на "Евровизия" "Врана" в короната
- неразказаната история на царския дворец А Слави можеше да стане депутат още 1994-а - 5-те удара, след които Трифонов
пак стърчи. А след шестия? Вижте първите страници на "24 часа" от 16 май през годините до днес. Заглавията са
интересни - спомнете си през какво преминаваше България. И сюжети, които обясняват защо сме там, където сме.

--- "24 часа" на 26 април - вижте първите страници през годините ---
Вижте първите страници на "24 часа" от 26 април през годините до днес. Заглавията са интересни - спомнете си п

In [144]:
mask = df["title"].str.startswith(
    (
        "Вижте отговора на фотозагадката:",
        "Фотозагадка:",
        "Виж отговора на фотозагадката:",
    ),
    na=False,
)
print(f"Articles: {mask.sum()}")

Articles: 12


In [145]:
df = drop_by_field(
    df,
    ['"24 часа" на 26 април'],
    field="title",
    match="contains",
)

Dropping 1 articles where title contains ['"24 часа" на 26 април']
Remaining articles: 50404


In [146]:
df = drop_by_field(
    df,
    [
        "Вижте отговора на фотозагадката:",
        "Фотозагадка:",
        "Виж отговора на фотозагадката:",
    ],
    field="title",
    match="startswith",
)

Dropping 12 articles where title startswith ['Вижте отговора на фотозагадката:', 'Фотозагадка:', 'Виж отговора на фотозагадката:']
Remaining articles: 50392


In [147]:
mask = df["full_text"].str.contains("Малкия Иванчо", na=False)
print(f"Articles containing 'Малкия Иванчо': {mask.sum()}\n")

for _, row in df[mask].iterrows():
    print(f"--- {row['source']} | {row['title']} ---")
    print(row["url"])
    print_by_words(row["full_text"])
    print()

Articles containing 'Малкия Иванчо': 18

--- 24chasa | Магия - виж оживялата карикатура на Ивайло Нинов ---
https://www.24chasa.bg/mneniya/article/22999378
Анимация карикатура на седмицата от автора на "Малкия Иванчо" Ивайло Нинов. Всяка седмица очаквайте нова карикатура с хумористичния поглед към
събитията, представен от Ивайло Нинов. Останалите карикатури вижте тук. Анимация карикатура на седмицата от автора на "Малкия Иванчо" Ивайло Нинов.
Всяка седмица очаквайте нова карикатура с хумористичния поглед към събитията, представен от Ивайло Нинов. Останалите карикатури вижте тук.

--- 24chasa | Кой и защо плаче в парламента - виж оживялата карикатура на Ивайло Нинов ---
https://www.24chasa.bg/mneniya/article/22973480
Анимация карикатура на седмицата от автора на "Малкия Иванчо" Ивайло Нинов. Всяка седмица очаквайте нова карикатура с хумористичния поглед към
събитията, представен от Ивайло Нинов. Останалите карикатури вижте тук. Анимация карикатура на седмицата от автора на "Малкия Иванч

In [148]:
df = drop_by_field(
    df, ["Малкия Иванчо"], field="full_text", source="24chasa", match="contains"
)

Dropping 18 articles where full_text contains ['Малкия Иванчо']
Remaining articles: 50374


**Finding:** Articles containing `Малкия Иванчо` are comic strips — they consist of image captions and dialogue with no factual news content.

**Decision:** Dropped all such records as they provide no meaningful information for clustering or summarization.


In [149]:
for phrase in ["Четете още", "Снимка:"]:
    mask = (df["source"] == "24chasa") & df["full_text"].str.contains(phrase, na=False)
    print(f"'{phrase}': {mask.sum()} articles\n")
    for _, row in df[mask].head(3).iterrows():
        print(f"--- {row['title']} ---")
        print(f"URL: {row['url']}")
        print_by_words(row["full_text"])
        print()

'Четете още': 620 articles

--- Само в "24 часа" на 28 май: Проучване на "Тренд" за доверието към кабинета "Радев" ---
URL: https://www.24chasa.bg/mneniya/article/22929159
Само в "24 часа" на 28 май: - Румен Радев смени стила на премиера - действа по устав. Интервю с
политолога Димитър Ганев - Търсят 30 милиарда евро от сивата икономика. Не социални пенсии, а помощи ще пестят на НОИ
182 млн. - Минимални заплати според професията - ще се отчита инфлацията и производителността - София като Мадрид - картата
за градския транспорт вече в телефона - Проверка в ДАНС: защо е спряна екстрадицията на собственика на незаконния град край
Варна - Къде да заведем децата си на 1 юни Вижте първите страници на "24 часа" от 28 май през
годините до днес. Заглавията са интересни - спомнете си през какво преминаваше България. И сюжети, които обясняват защо сме там,
където сме. Четете още - Само в "24 часа" на 27 май: Рисковете от наводнения са описани, но не и
как се действа, ако ни залеят - Само в "24 час

In [150]:
phrase_pattern = re.compile(r"\sЗатвори\s+[А-Я]")

In [151]:
mask = (df["source"] == "24chasa") & df["full_text"].str.contains(
    phrase_pattern, na=False
)
print(f"Articles containing 'Затвори' followed by uppercase: {mask.sum()}\n")

for _, row in df[mask].head(20).iterrows():
    match = phrase_pattern.search(row["full_text"])
    if match:
        idx = match.start()
        total_len = len(row["full_text"])
        position_pct = idx / total_len * 100
        after = row["full_text"][idx:].strip()
        print(f"--- {row['title']} ---")
        print(f"Position: {position_pct:.0f}% into text")
        print(f"{after[:200]}")
        print()

Articles containing 'Затвори' followed by uppercase: 619

--- Цената на петрола на ОПЕК вече е под 97 долара за барел ---
Position: 82% into text
Затвори Цената на петрола на ОПЕК падна под 97 долара за барел Снимка: СНИМКА: Pixabay (илюстративна) Четете още Петролът на ОПЕК падна под 107 долара за барел ОПЕК понижи прогнозата за световното тър

--- Сигнали за бомби са получени в училищата в Хърватия, Босна и Херцеговина и Черна гора ---
Position: 92% into text
Затвори Снимка: Снимка: Архив Четете още Сигнали за бомби в 4 училища в София Фалшиви сигнали за бомби опразниха училища в Черна гора и Босна и Херцеговина В училища в цяла Сърбия са получени заплахи 

--- Изтребители се сблъскали в Южна Корея, защото пилотите им снимали ---
Position: 74% into text
Затвори Изтребител СНИМКА: Георги Кюрпанов - Генк (Снимката е илюстративна) Четете още Руски изтребител се разби при тренировъчен полет над Крим Румънски изтребител се разби по време на съвместно учен

--- Дeлът на китайските марки ав

In [152]:
def truncate_at_pattern(text: str, pattern) -> str:
    if not isinstance(text, str):
        return text
    match = pattern.search(text)
    if match:
        return text[: match.start()].strip()
    return text

In [153]:
df = apply_to_source(
    df, lambda t: truncate_at_pattern(t, phrase_pattern), source="24chasa"
)

In [154]:
print_source_samples(df, "24chasa", n=3)

=== 24CHASA (12969 articles) ===

--- Спасителят на украинци край Малко Търново: Бетонна стена е спряла рейса, без нея нямаше да има оцелели ---
URL: https://www.24chasa.bg/bulgaria/article/22828636
- Единствената мисъл на граничния полицай Ивелин Арабаджиев, докато вадел хората от автобуса, била да не гръмне работещият двигател
- Преди 10 г. той помага на премръзнали мигранти, сред които и 3-годишни деца
Още един достоен българин влиза в редиците на кампанията на вестник "24 часа" - това е младши експерт Ивелин Арабаджиев, командир на отделение във Втора група за охрана на държавната граница в ГПУ-Малко Търново. На 22 април т.г. с риск за собствения си живот той спасява пътници от украински туристически автобус, който потегля неуправляемо на собствен ход и пропада в дере.
При инцидента загиват двама души, но жертвите вероятно щяха да са много повече, ако не бе намесата на младия граничен полицай.
За достойната си постъпка полицай Арабаджиев бе награден
от служебния министър на МВР Еми

### 24chasa — Cleaning Summary

- Dropped daily road condition reports (`Вижте какво е състоянието на пътищата`) — structured government bulletins, not news articles
- Dropped photo puzzle articles (`Фотозагадка:`, `Виж/Вижте отговора на фотозагадката:`)
- Dropped comic strip articles containing `Малкия Иванчо`
- Dropped boilerplate app promotion articles (`изтегли приложението от тук` in title)
- Dropped articles with title containing `"24 часа" на` (daily digest covers)
- Dropped daily TV schedule articles (`Спорт по телевизията днес`)
- Stripped newsletter prompt appearing before article content (`Разберете големите новини от последното денонощие накуп Запиши се`)
- Removed inline newsletter subscription block (`"24 часа" в мейла ти. Разберете големите новини от последното денонощие накуп`)
- Truncated everything after `Последвайте ни в Google News Showcase` (subscription/ad block always at end)
- Truncated everything after `Затвори [Uppercase]` pattern (UI navigation artifact injected at end of text)
- Dropped lines starting with `Снимка:` (photo captions)
- Dropped lines containing only Latin characters (embedded tweets, foreign text)
- Dropped social media attribution lines
- Dropped emoji-heavy and `@handle` lines (embedded social media posts)


## 4.k. Fakti

In [155]:
print_source_samples(df, "fakti", n=3, random_state=42)

=== FAKTI (7583 articles) ===

--- Щедри заплати и трикове: Как вербуват руснаци за фронта ---
URL: https://fakti.bg/world/1058740-shtedri-zaplati-i-trikove-kak-verbuvat-rusnaci-za-fronta
Поради липсата на достатъчно желаещи да се бият на фронта, руските вербовчици измислят нови начини да примамват нови попълнения във войната срещу Украйна. Руското издание "Верстка" е открило обяви, в които се предлага на желаещи да сключат договор за служба "в тила на специалната военна операция (СВО)" в Беларус или Китай. В една от тях се обещава: "Няма да има никакви командировки в горещите точки и по линиите на бойно съприкосновение". Уточнява се, че ситуацията в Беларус е стабилна, има развита инфраструктура, а за много руски граждани мястото не е отдалечено от дома им. Освен това става ясно, че военната служба ще се извършва на територията на съюзната държава с гаранции и заплащане от Министерството на отбраната на Руската федерация.
Според "Верстка" през май 2026 г. в платформата „Авито“ са били

In [156]:
df = apply_to_source(
    df,
    lambda t: truncate_at_sentinel(t, ["Поставете оценка:", "Напиши коментар:"]),
    source="fakti",
)

### Fakti — Cleaning Summary

- Truncated everything after `Поставете оценка:` or `Напиши коментар:` — these mark the start of the comment section appended to the article by the site, containing UI boilerplate with no news content


## 5. General Noise Removal

In [157]:
noise_lines = []

for _, row in df.iterrows():
    if not isinstance(row["full_text"], str):
        continue
    for line in row["full_text"].split("\n"):
        if is_noise_line(line):
            noise_lines.append(
                {"source": row["source"], "title": row["title"], "line": line.strip()}
            )

print(f"Total noise lines found: {len(noise_lines)}\n")

for item in noise_lines[:20]:
    print(f"[{item['source']}] {item['title']}")
    print(f"  >> {item['line']}")
    print()

Total noise lines found: 1619

[monitor] Гледайте международните приятелски мачове
  >> Diema Sport

[monitor] Гледайте международните приятелски мачове
  >> Diema Sport 2

[monitor] Гледайте международните приятелски мачове
  >> Diema Sport 3

[monitor] Гледайте международните приятелски мачове
  >> Nova Sport

[monitor] Гледайте международните приятелски мачове
  >> MAX Sport 1

[monitor] Гледайте международните приятелски мачове
  >> MAX Sport 2

[monitor] Гледайте международните приятелски мачове
  >> MAX Sport 3

[monitor] Гледайте международните приятелски мачове
  >> MAX Sport 4

[monitor] Гледайте международните приятелски мачове
  >> EUROSPORT

[monitor] СЛЕД ЖЕСТОКОТО ЗЕМЕТРЕСЕНИЕ: Жертвите във Филипините са над 40! (ВИДЕО)
  >> More wild footage from today’s massive earthquake in the Philippines. pic.twitter.com/Cdylad2d7W

[monitor] СЛЕД ЖЕСТОКОТО ЗЕМЕТРЕСЕНИЕ: Жертвите във Филипините са над 40! (ВИДЕО)
  >> — Breaking911 (@Breaking911) June 8, 2026

[monitor] ХОЛИВУДСКА МО

In [158]:
df["full_text"] = df["full_text"].apply(lambda t: drop_lines(t, is_noise_line))

### Global Noise Line Removal

Applied across the entire dataset — dropped lines matching any of the following patterns:

- Lines containing only Latin characters (English text, foreign-language content)
- Lines starting with `—`/`–` followed by Latin content (social media quotes, foreign attributions)
- Lines starting with `@handle` (embedded TikTok/Twitter posts)
- Social media attribution lines matching `(@handle) date` pattern
- Emoji-heavy lines


## Removing inline separators

In [159]:
mask = df["full_text"].str.contains(r"(?:-\s*){5,}", regex=True, na=False)
print(f"Articles with inline separators: {mask.sum()}")
print(df[mask]["source"].value_counts().to_string())
print()

for _, row in df[mask].head(5).iterrows():
    print(f"--- [{row['source']}] {row['title']} ---")
    print_by_words(row["full_text"])
    print()

Articles with inline separators: 27
source
bta             9
fakti           8
segabg          4
actualno        3
24chasa         1
standartnews    1
monitor         1

--- [24chasa] България спечели отборната титла на европейското по художествена гимнастика във Варна ---
България спечели отборната титла при жените на Европейското първенство по художествена гимнастика във Варна, а ансамбълът зае шесто място в
многобоя. Националките Стилияна Николова, Ева Брезалиева при жените и София Иванова, Маргарита Василева, Магдалина Миневска, Емилия Обретенова, Рая Божилова и
Магдалена Вълкова събраха от десетте си изигравания 277.600 точки. На второ място е Израел с 274.650 точки, а на трето
Русия с 274.650 точки. В топ 6 са още Испания с 274.600, Германия с 269.600, Италия с 269.150 и други.
България е отборен шампион при жените през 2022, 2023, 2024, единствено миналата година отстъпи пред Италия, а сега отново е
номер 1 на Стария континент. Европейското първенство е квалификация за Световнот

In [160]:
def remove_inline_separators(text: str) -> str:
    if not isinstance(text, str):
        return text
    return re.sub(r"(?:-\s*){5,}", " ", text).strip()

In [161]:
df["full_text"] = df["full_text"].apply(remove_inline_separators)

## Removing HTML entities

In [162]:
mask = df["full_text"].str.contains(r"&\w+;", regex=True, na=False)
print(f"Articles with HTML entities: {mask.sum()}")
print(df[mask]["source"].value_counts().to_string())

Articles with HTML entities: 13
source
actualno    13


In [163]:
mask = (df["source"] == "actualno") & df["full_text"].str.contains(
    r"&\w+;", regex=True, na=False
)
print(f"Actualno articles with HTML entities: {mask.sum()}\n")

for _, row in df[mask].iterrows():
    print(f"--- {row['title']} ---")
    print_by_words(row["full_text"])
    print()

Actualno articles with HTML entities: 13

--- Почти 12 милиона евро субсидия: Кой колко получава и кои са големите губещи? (СНИМКИ) ---
Почти 12 милиона евро субсидия: Кой колко получава и кои са големите губещи? (СНИМКИ) 22 април 2026, 16:31 часа 1675
прочитания 0 коментара Снимка: БГНЕС 11 милиона и 731 хиляди евро - това е размерът на държавната субсидия, която ще
си разпределят партиите и коалициите след предсрочните парламентарни избори на 19 април, сочат изчисленията на Actualno.com според предварителните резултати от
вота. От тях най-голямата сума респективно ще отиде за "Прогресивна България" - над 5,924 милиона евро. Как се изчислява -
всяка партия, събрала над 1% подкрепа, получава субсидия. За коалициите прагът е 4%. Ситуацията с резултатите е доста специфична на
тези избори и показва, че това е ироничен урок за част от формациите, които се явиха, но покриват едва половината
критерии. Такъв е случая с БСП и АПС, които имат над 1%, но се явиха като коалиции и затова остават
б

In [164]:
def decode_html_entities(text: str) -> str:
    if not isinstance(text, str):
        return text
    prev = None
    while prev != text:
        prev = text
        text = html.unescape(text)
    return text


df["full_text"] = df["full_text"].apply(decode_html_entities)

In [165]:
# Word count recheck after all cleaning
df["word_count"] = df["full_text"].fillna("").apply(lambda x: len(x.split()))
short = df[df["word_count"] < 30]
print(f"Articles under 30 words after cleaning: {len(short)}")
print(short[["source", "title", "word_count"]].to_string())

Articles under 30 words after cleaning: 178
             source                                                                                                                                                          title  word_count
1513            bta                    За учебната 2025/2026 година чуждестранните студенти в Софийския университет са 1431 и са от 55 държави, съобщиха за БТА от висшето училище           3
2860            bta                       Четири са вече жертвите на вчерашната катастрофа между автобус и две коли в София; скоростта на автомобилите е била между 104 и 150 км/ч          24
3604            bta  Двама души загубиха живота си в катастрофа с две леки коли и автобус на градския транспорт в София, шофьорите на леките автомобили са се движили със 150 км/ч          28
6075            bta      Предложението за асоциирано членство на Украйна е възможност, която Киев трябва да използва, каза евродепутатът Дейвид Макалистър пред Европейския нюзрум          26
7

In [166]:
short_mask = df["word_count"] < 30
print(f"Articles under 30 words across whole dataset: {short_mask.sum()}\n")

for _, row in df[short_mask].iterrows():
    print(f"--- {row['title']} ({row['word_count']} words) ---")
    print(row["url"])
    print(row["full_text"])
    print()

Articles under 30 words across whole dataset: 178

--- За учебната 2025/2026 година чуждестранните студенти в Софийския университет са 1431 и са от 55 държави, съобщиха за БТА от висшето училище (3 words) ---
https://www.bta.bg/bg/news/lik/1142704-za-uchebnata-2025-2026-godina-chuzhdestrannite-studenti-v-sofiyskiya-universitet
За учебната 2025

--- Четири са вече жертвите на вчерашната катастрофа между автобус и две коли в София; скоростта на автомобилите е била между 104 и 150 км/ч (24 words) ---
https://www.bta.bg/bg/news/bulgaria/1142045-chetiri-sa-veche-zhertvite-na-vcherashnata-katastrofa-mezhdu-avtobus-i-dve-koli-
Четири са вече жертвите на вчерашната катастрофа между автобус и две коли в София; скоростта на автомобилите е била между 104 и 150 км

--- Двама души загубиха живота си в катастрофа с две леки коли и автобус на градския транспорт в София, шофьорите на леките автомобили са се движили със 150 км/ч (28 words) ---
https://www.bta.bg/bg/news/bulgaria/1141760-dvama-dushi-zag

In [167]:
df = df[df["word_count"] >= 30].reset_index(drop=True)
print(f"Remaining articles: {len(df)}")

Remaining articles: 50196


## Checking for duplicated paragraphs

In [168]:
def has_duplicate_paragraphs(text: str) -> bool:
    if not isinstance(text, str):
        return False
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    return len(paragraphs) != len(set(paragraphs))


mask = df["full_text"].apply(has_duplicate_paragraphs)
print(f"Articles with duplicate paragraphs: {mask.sum()}")
print(df[mask]["source"].value_counts().to_string())
print()

for _, row in df[mask].head(10).iterrows():
    print(f"--- {row['title']} ({row['word_count']} words) ---")
    print(f"URL: {row['url']}")
    print_by_words(row["full_text"])
    print()

Articles with duplicate paragraphs: 103
source
24chasa    103

--- Движението в тунел "Топли дол" към Варна бе в активната лента заради разлято гориво (72 words) ---
URL: https://www.24chasa.bg/spravochnik/article/22999268
Временно движението в тунел "Топли дол" на АМ "Хемус" в посока Варна се осъществява в активната лента поради разлято гориво
на пътното платно, съобщиха от АПИ. Трафикът се регулира от "Пътна полиция". По-късно движението бе възстановено. Временно движението в тунел
"Топли дол" на АМ "Хемус" в посока Варна се осъществява в активната лента поради разлято гориво на пътното платно, съобщиха
от АПИ. Трафикът се регулира от "Пътна полиция". По-късно движението бе възстановено.

--- Катастрофа при км 183 от пътя Казанлък - Габрово, движението е в една лента (76 words) ---
URL: https://www.24chasa.bg/spravochnik/article/22999350
Временно движението при км 183 по път I-5 Казанлък - Габрово се осъществява двупосочно в една лента поради ПТП, съобщиха
от АПИ. Шофьорите да се дви

In [169]:
def remove_duplicate_paragraphs(text: str) -> str:
    if not isinstance(text, str):
        return text
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    seen = []
    for p in paragraphs:
        if p not in seen:
            seen.append(p)
    return "\n\n".join(seen)

In [170]:
mask = df["full_text"].apply(has_duplicate_paragraphs)
print(f"Articles with duplicate paragraphs before cleaning: {mask.sum()}")

df["full_text"] = df["full_text"].apply(remove_duplicate_paragraphs)

mask = df["full_text"].apply(has_duplicate_paragraphs)
print(f"Articles with duplicate paragraphs after cleaning: {mask.sum()}")

Articles with duplicate paragraphs before cleaning: 103
Articles with duplicate paragraphs after cleaning: 0


In [171]:
row = df[df["url"] == "https://www.24chasa.bg/bulgaria/article/22853733"].iloc[0]
print_by_words(row["full_text"])

Георги Тодоров е назначен за заместник-министър на туризма. "Със заповед на министър-председателя Румен Радев на длъжността заместник-министър на туризма е
назначен Георги Георгиев Тодоров", съобщават от правителствената информационна служба.


## 6. Title cleaning

In [172]:
mask = df["title"].str.contains(r"<[a-zA-Z]", regex=True, na=False)
print(f"Titles with HTML: {mask.sum()}")
print(df[mask]["source"].value_counts().to_string())
print()

for _, row in df[mask].head(10).iterrows():
    print(f"[{row['source']}] {row['title']}")

Titles with HTML: 515
source
vesti    515

[vesti] <p>Ким Чен-ун и Си Дзинпин с ключова среща в Пхенян, какво договориха</p>
[vesti] <p>Зеленски с &bdquo;желязна&ldquo; подкрепа от Чарлз III и покана за историческа визита</p>
[vesti] <p>Фалстарт на делото за побоя над директора на полицията в Русе</p>
[vesti] <p>Кой е новият и.ф. главен секретар на МВР Любомир Николов</p>
[vesti] <p>Опит за селфи с мечка&nbsp;прати трима туристи в болница</p>
[vesti] <p>Капанът на съюзите поглъща Доналд Тръмп</p>
[vesti] <p>Пътнически самолет кацна аварийно заради пукнато предно стъкло</p>
[vesti] <p>Отстраниха главния прокурор на МНС заради обвинения в сексуално посегателство</p>
[vesti] <p>Ето кой ще заеме временно поста на Георги Кандев</p>
[vesti] <p>11-метрови вълни удариха Уелингтън, стотици бяха евакуирани</p>


In [173]:
df["title"] = df["title"].apply(strip_html)
df["title"] = df["title"].apply(decode_html_entities)

In [174]:
short_titles = df[df["title"].str.split().str.len() < 3]
print(f"Titles under 3 words: {len(short_titles)}\n")

for _, row in short_titles.iterrows():
    print(f"--- [{row['source']}] {row['title']} ---")
    print(f"URL: {row['url']}")
    print_by_words(row["full_text"])
    print()

Titles under 3 words: 20

--- [actualno] Диагноза "Тартюф" ---
URL: https://www.actualno.com/scene/diagnoza-tartuf-news_2604626.html
Всичко и всички са фейк - гòрко оплаква GenZ своето време! Как да не са, като социални (ко)медии ежедневно изтъпанчват
на сцена. Ежедневно пред онлайн публика. При това не само GenZ, а все по-често мама, тате, бате, баба, дядо… Световъртеж
в техно водовъртеж. Спокойствие? Лично пространство? Скрупули? Ех – отлетяха…! Докато залисан си в играта, роля погуби… същината. Нещо си
отива. Друго късо дава. Човешкото на късо – маската продава! Роден актьор е човек – нали прякор му е homo
ludens. Роден актьор за невидим режисьор. Лесно поддава. Играе - не роптае. Влезе ли в роля, понякога трябва да иде
на театър, за да се събуди как да излезе. Да иде на театър, за да… избегне психиатър. Гледам "Тартюф или
лицемерът". Лилия Абаджиева поставя бароковата комедия на Жан-Батист Молиер в Малък градски театър "Зад канала". Както се убеждавам малко по-късно,
представление 

### Title Cleaning Summary

- Stripped HTML tags from titles (all `<p>` wrappers from `vesti`)
- Decoded HTML entities (e.g. `&quot;` → `"`, `&bdquo;` → `„`, `&ldquo;` → `"`)
- Short titles are valid


In [176]:
for source, threshold in [("24chasa", 3000), ("fakti", 2000)]:
    outliers = df[
        (df["source"] == source) & (df["word_count"] > threshold)
    ].sort_values("word_count", ascending=False)
    print(f"=== {source} > {threshold} words: {len(outliers)} ===")
    for _, row in outliers.head(5).iterrows():
        print(f"  [{row['word_count']} words] {row['title']}")
        print(f"  URL: {row['url']}")
        print_by_words(row["full_text"])
        print()

=== 24chasa > 3000 words: 18 ===
  [13275 words] Вижте ситуацията по пътищата на страната в момента
  URL: https://www.24chasa.bg/spravochnik/article/22910525
Ето информация от "Пътна структура" за състоянието на шосетата в страната. I.МЕТЕОРОЛОГИЧНА ОБСТАНОВКА: През първата половина на нощта валежи ще
има главно в планинските райони. До сутринта ще отслабнат и ще спрат. Облачността в повечето места ще се разкъса и
намалее. В източните райони ще е със слаб северозападен вятър. В останалата част от страната вятърът ще стихне и главно
в западната част от Дунавската равнина ще има ниска слоеста облачност и намалена видимост. Утре преди обяд ще бъде предимно
слънчево. Около и след обяд ще се развива купеста облачност, но само на отделни места, главно в планинските райони ще
превали краткотраен дъжд. Вятърът ще е слаб от север-северозапад, в Източна България до умерен от север-североизток. Максималните температури в повечето
места ще бъдат между 22° и 27°, в София – около 24°. ОГРАНИЧЕНИЯ З

There are still outliers for words distribution, but they are valid texts, so all good 

In [177]:
len(df)

50196

## 7. Media-type suffixes in titles and full_text

Check how many articles carry `(видео)`, `(снимка)`, `(снимки)`, `(снимка+видео)`, `(видео+снимка)` — appended by CMS editors as metadata tags, not actual article content.

In [178]:
MEDIA_PATTERNS = ["(видео)", "(снимка)", "(снимки)", "(снимка+видео)", "(видео+снимка)", "(видео+снимки)", "(снимки, видео)"]

def media_re(p):
    # Allow optional spaces around + so (снимка+видео) matches (снимка + видео) too
    return re.escape(p).replace(r"\+", r"\s*\+\s*")

combined = "|".join(media_re(p) for p in MEDIA_PATTERNS)

# Per-pattern counts
for field in ["title", "full_text"]:
    print(f"--- {field} ---")
    for p in MEDIA_PATTERNS:
        n = df[field].str.contains(media_re(p), case=False, na=False, regex=True).sum()
        if n:
            print(f"  {p:<22}: {n}")
    total = df[field].str.contains(combined, case=False, na=False, regex=True).sum()
    print(f"  {'TOTAL':<22}: {total}")
    print()

--- title ---
  (видео)               : 1536
  (снимка)              : 112
  (снимки)              : 536
  (снимка+видео)        : 1
  (видео+снимка)        : 12
  (видео+снимки)        : 98
  (снимки, видео)       : 23
  TOTAL                 : 2318

--- full_text ---
  (видео)               : 1780
  (снимка)              : 92
  (снимки)              : 440
  (снимка+видео)        : 2
  (видео+снимка)        : 9
  (видео+снимки)        : 128
  (снимки, видео)       : 6
  TOTAL                 : 2317



In [179]:
# Per-source breakdown
for field in ["title", "full_text"]:
    mask = df[field].str.contains(combined, case=False, na=False, regex=True)
    print(f"{field} ({mask.sum()} affected):")
    print(df[mask]["source"].value_counts().to_string())
    print()

title (2318 affected):
source
actualno        1059
24chasa          389
nova             364
fakti            256
monitor          193
standartnews      37
vesti             20

full_text (2317 affected):
source
actualno    1727
nova         426
monitor      102
vesti         35
fakti         17
24chasa       10



In [180]:
def strip_media_suffixes(text: str) -> str:
    if not isinstance(text, str):
        return text
    return re.sub(combined, "", text, flags=re.IGNORECASE).strip()

In [181]:
df["title"] = df["title"].apply(strip_media_suffixes)
df["full_text"] = df["full_text"].apply(strip_media_suffixes)

In [182]:
mask = df["title"].str.contains(combined, case=False, na=False, regex=True)
print(f"Remaining titles with media suffixes: {mask.sum()}")

Remaining titles with media suffixes: 0


In [183]:
mask = df["full_text"].str.contains(combined, case=False, na=False, regex=True)
print(f"Remaining fill_text with media suffixes: {mask.sum()}")

Remaining fill_text with media suffixes: 0


## 8.Updated articles posted in the same time

In [184]:
def inspect_source_duplicate_similarity(
    df: pd.DataFrame, source: str, threshold: float = 0.5
):
    source_df = df[df["source"] == source].copy()
    groups_found = 0
    high_title_sim = 0
    high_text_sim = 0

    for published_at, group in source_df.groupby("published_at_dt"):
        if len(group) < 2:
            continue

        titles = group["title"].fillna("").tolist()
        texts = group["full_text"].fillna("").tolist()
        urls = group["url"].tolist()
        fetched = group["fetched_at"].tolist()

        try:
            title_sim = cosine_similarity(TfidfVectorizer().fit_transform(titles))[0][1]
            text_sim = cosine_similarity(TfidfVectorizer().fit_transform(texts))[0][1]
        except ValueError:
            continue

        groups_found += 1
        if title_sim >= threshold:
            high_title_sim += 1
        if text_sim >= threshold:
            high_text_sim += 1

        if title_sim < threshold and text_sim < threshold:
            continue

        print(
            f"Published: {published_at} | Title sim: {title_sim:.3f} | Text sim: {text_sim:.3f}"
        )
        for t, u, f in zip(titles, urls, fetched):
            print(f"  [{f}] {t}")
            print(f"    {u}")
        print()

    print(
        f"Source: {source} | Total groups: {groups_found} | Title sim >= {threshold}: {high_title_sim} | Text sim >= {threshold}: {high_text_sim}"
    )

In [185]:
same_time_all = df.groupby(["source", "published_at_dt"]).filter(lambda x: len(x) > 1)
summary = same_time_all.groupby("source")["published_at_dt"].count()
print("Articles with duplicate timestamps per source:")
print(summary.sort_values(ascending=False))

Articles with duplicate timestamps per source:
source
24chasa         2702
actualno        1250
nova             676
vesti            353
bta              316
segabg           281
standartnews      30
fakti              2
Name: published_at_dt, dtype: int64


In [186]:
inspect_source_duplicate_similarity(df, "fakti", threshold=0.7)

Source: fakti | Total groups: 1 | Title sim >= 0.7: 0 | Text sim >= 0.7: 0


In [187]:
inspect_source_duplicate_similarity(df, "segabg", threshold=0.7)

Published: 2026-05-04 12:38:11+00:00 | Title sim: 0.570 | Text sim: 1.000
  [2026-05-05T10:07:18.707630+00:00] 47-годишна литовка намушка четирима във Варна
    https://www.segabg.com/hot/category-bulgaria/47-godishna-litovka-namushka-chetirima-vuv-varna
  [2026-05-04T19:45:07.875690+00:00] 47-годишна жена от Литва намушка четирима души във Варна
    https://www.segabg.com/hot/category-bulgaria/47-godishna-zhena-litva-namushka-chetirima-dushi-vuv-varna

Published: 2026-05-04 17:14:05+00:00 | Title sim: 1.000 | Text sim: 1.000
  [2026-05-05T02:55:05.912365+00:00] Бившият шеф на пациентска организация излиза на свобода срещу 60 000 евро
    https://www.segabg.com/category-bulgaria/bivshiyat-shef-na-pacientska-organizaciya-izliza-na-svoboda-sreshtu-60-000-evro
  [2026-05-04T19:45:07.875658+00:00] Бившият шеф на пациентска организация излиза на свобода срещу 60 000 евро
    https://www.segabg.com/hot/category-bulgaria/bivshiyat-shef-na-pacientska-organizaciya-izliza-na-svoboda-sreshtu-60-0

In [188]:
inspect_source_duplicate_similarity(df, "bta", threshold=0.7)

Published: 2026-05-05 15:05:00+00:00 | Title sim: 1.000 | Text sim: 0.883
  [2026-05-05T16:13:48.135656+00:00] Президентът Илияна Йотова ще връчи мандата за съставяне на правителство на 7 май
    https://www.bta.bg/bg/news/bulgaria/1120064-prezidentat-iliyana-yotova-shte-vrachi-mandata-za-sastavyane-na-pravitelstvo-na-
  [2026-05-05T15:32:43.295435+00:00] Президентът Илияна Йотова ще връчи мандата за съставяне на правителство на 7 май
    https://www.bta.bg/bg/news/bulgaria/1120042-prezidentat-iliyana-yotova-shte-vrachi-mandata-za-sastavyane-na-pravitelstvo-na-

Published: 2026-05-07 14:10:00+00:00 | Title sim: 1.000 | Text sim: 0.667
  [2026-05-07T14:18:48.663588+00:00] Кандидатът за премиер от "Прогресивна България" Румен Радев представи структура и състав на проектокабинет
    https://www.bta.bg/bg/news/bulgaria/1120749-kandidatat-za-premier-ot-progresivna-balgariya-rumen-radev-predstavi-struktura
  [2026-05-07T14:18:48.663519+00:00] Кандидатът за премиер от "Прогресивна България" Р

In [189]:
inspect_source_duplicate_similarity(df, "vesti", threshold=0.7)

Published: 2026-05-05 03:45:00+00:00 | Title sim: 0.063 | Text sim: 0.833
  [2026-05-05T10:07:47.065231+00:00] "Прогресивна България": До края на седмицата ще има правителство
    https://www.vesti.bg/bulgaria/progresivna-bylgariiado-kraia-na-sedmicata-shte-ima-pravitelstvo-6258729
  [2026-05-05T06:20:11.295865+00:00] Старт на консултациите: Президентът се среща с парламентарните групи
    https://www.vesti.bg/bulgaria/start-na-konsultaciite-prezidentytse-sreshta-s-parlamentarnite-grupi-6258729

Published: 2026-05-05 09:10:00+00:00 | Title sim: 0.580 | Text sim: 0.911
  [2026-05-05T15:33:44.268320+00:00] Две земетресения разлюляха Сърбия, едното е причинило сериозни щети
    https://www.vesti.bg/sviat/dve-zemetreseniia-razliuliaha-syrbiia-ednoto-e-prichinilo-seriozni-shteti-6258762
  [2026-05-05T10:07:47.064850+00:00] Две земетресения разлюляха Сърбия
    https://www.vesti.bg/sviat/dve-zemetreseniia-razliuliaha-syrbiia-6258762

Published: 2026-05-05 18:09:00+00:00 | Title sim: 0.669 | 

In [190]:
inspect_source_duplicate_similarity(df, "nova", threshold=0.7)

Published: 2026-04-23 05:00:25+00:00 | Title sim: 0.802 | Text sim: 0.992
  [2026-04-23T09:47:56.601934+00:00] Високи температури, следвани от бури с градушки, през следващите дни
    https://nova.bg/news/view/2026/04/23/534810/%D0%B2%D0%B8%D1%81%D0%BE%D0%BA%D0%B8-%D1%82%D0%B5%D0%BC%D0%BF%D0%B5%D1%80%D0%B0%D1%82%D1%83%D1%80%D0%B8-%D1%81%D0%BB%D0%B5%D0%B4%D0%B2%D0%B0%D0%BD%D0%B8-%D0%BE%D1%82-%D0%B1%D1%83%D1%80%D0%B8-%D1%81-%D0%B3%D1%80%D0%B0%D0%B4%D1%83%D1%88%D0%BA%D0%B8-%D0%BF%D1%80%D0%B5%D0%B7-%D1%81%D0%BB%D0%B5%D0%B4%D0%B2%D0%B0%D1%89%D0%B8%D1%82%D0%B5-%D0%B4%D0%BD%D0%B8/
  [2026-04-23T06:08:26.648955+00:00] Летни температури, следвани от бури с градушки, през следващите дни
    https://nova.bg/news/view/2026/04/23/534810/%D0%BB%D0%B5%D1%82%D0%BD%D0%B8-%D1%82%D0%B5%D0%BC%D0%BF%D0%B5%D1%80%D0%B0%D1%82%D1%83%D1%80%D0%B8-%D1%81%D0%BB%D0%B5%D0%B4%D0%B2%D0%B0%D0%BD%D0%B8-%D0%BE%D1%82-%D0%B1%D1%83%D1%80%D0%B8-%D1%81-%D0%B3%D1%80%D0%B0%D0%B4%D1%83%D1%88%D0%BA%D0%B8-%D0%BF%D1%80%D0%B5%D0%B7-

In [191]:
inspect_source_duplicate_similarity(df, "actualno", threshold=0.7)

Published: 2026-04-21 11:11:00+00:00 | Title sim: 0.317 | Text sim: 0.825
  [2026-04-21T13:49:43.207655+00:00] Крум Зарков поиска вот на доверие и обяви мащабен процес на промяна в БСП
    https://www.actualno.com/politics/krum-zarkov-poiska-vot-na-doverie-i-objavi-mashtaben-proces-na-promjana-v-bsp-news_2584392.html
  [2026-04-21T11:12:12.529337+00:00] Оставката чака зад ъгъла: Крум Зарков иска вот на доверие от БСП след "много лошия" резултат
    https://www.actualno.com/politics/ostavkata-chaka-zad-ygyla-krum-zarkov-iska-vot-na-doverie-ot-bsp-sled-mnogo-loshija-rezultat-news_2584392.html

Published: 2026-04-21 21:35:00+00:00 | Title sim: 0.868 | Text sim: 0.994
  [2026-04-22T06:03:40.205382+00:00] Мбапе и Винисиус са във вихъра си: Реал Мадрид се доближи до Барселона в Ла Лига
    https://www.actualno.com/football/mbape-i-vinisius-sa-vyv-vihyra-si-real-madrid-se-dobliji-do-barselona-v-la-liga-news_2584657.html
  [2026-04-22T02:48:16.526036+00:00] Мбапе и Винисиус са във вихъра си: Р

In [192]:
inspect_source_duplicate_similarity(df, "24chasa", threshold=0.7)

Published: 2026-04-22 16:49:00+00:00 | Title sim: 1.000 | Text sim: 0.993
  [2026-04-22T20:58:34.871000+00:00] Испански прокурор поиска делото за корупция срещу съпругата на премиера да бъде прекратено
    https://www.24chasa.bg/mezhdunarodni/article/22716371
  [2026-04-22T17:05:17.998300+00:00] Испански прокурор поиска делото за корупция срещу съпругата на премиера да бъде прекратено
    https://www.24chasa.bg/zdrave/article/22716371

Published: 2026-04-23 14:33:00+00:00 | Title sim: 1.000 | Text sim: 1.000
  [2026-04-23T20:56:14.066825+00:00] Правителството на Тръмп улеснява употребата на марихуана за медицински цели
    https://www.24chasa.bg/zdrave/article/22722225
  [2026-04-23T15:56:53.425387+00:00] Правителството на Тръмп улеснява употребата на марихуана за медицински цели
    https://www.24chasa.bg/mezhdunarodni/article/22722225

Published: 2026-04-23 16:43:00+00:00 | Title sim: 1.000 | Text sim: 1.000
  [2026-04-23T19:15:06.157799+00:00] Арестуваха бившия шеф на северномакедон

In [193]:
def drop_updated_duplicates(
    df: pd.DataFrame, sources: list[str], threshold: float = 0.7
) -> pd.DataFrame:
    df = df.sort_values("fetched_at", ascending=False).reset_index(drop=True)
    rows_to_drop = set()

    for source in sources:
        source_df = df[df["source"] == source]
        source_dropped = 0

        for published_at, group in source_df.groupby("published_at_dt"):
            if len(group) < 2:
                continue

            titles = group["title"].fillna("").tolist()
            texts = group["full_text"].fillna("").tolist()

            try:
                title_sim = cosine_similarity(TfidfVectorizer().fit_transform(titles))[
                    0
                ][1]
                text_sim = cosine_similarity(TfidfVectorizer().fit_transform(texts))[0][
                    1
                ]
            except ValueError:
                continue

            if title_sim < threshold and text_sim < threshold:
                continue

            indices = group.index.tolist()
            to_drop = indices[1:]
            rows_to_drop.update(to_drop)
            source_dropped += len(to_drop)

        print(f"Source: {source} | Dropping: {source_dropped}")

    before = len(df)
    df = df.drop(index=rows_to_drop).reset_index(drop=True)
    print(f"\nTotal dropped: {before - len(df)} | Remaining: {len(df)}")
    return df

In [194]:
sources_to_clean = ["segabg", "bta", "vesti", "nova", "actualno"]
df = drop_updated_duplicates(df, sources_to_clean)

Source: segabg | Dropping: 145
Source: bta | Dropping: 27
Source: vesti | Dropping: 46
Source: nova | Dropping: 320
Source: actualno | Dropping: 46

Total dropped: 584 | Remaining: 49612


Update the parquet file

In [195]:
out_path = "../data/processed/articles_clean_notebook.parquet"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df.to_parquet(out_path, index=False)
print(f"Saved {len(df)} articles → {out_path}")

Saved 49612 articles → ../data/processed/articles_clean_notebook.parquet


# 9. Comparison with raw data

In [196]:
DATA_DIR = "../data/raw"

records = []
for date_folder in sorted(os.listdir(DATA_DIR)):
    folder_path = os.path.join(DATA_DIR, date_folder)
    if not os.path.isdir(folder_path):
        continue
    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            filepath = os.path.join(folder_path, filename)
            with open(filepath, encoding="utf-8") as f:
                records.append(json.load(f))

df_raw = pd.DataFrame(records)
df_raw["published_at_dt"] = pd.to_datetime(
    df["published_at"], format="ISO8601", utc=True
)
print(f"Total articles loaded: {len(df_raw)}")

Total articles loaded: 63178


In [197]:
comp = (
    pd.DataFrame(
        {"raw": df_raw["source"].value_counts(), "cleaned": df["source"].value_counts()}
    )
    .fillna(0)
    .astype(int)
)
comp["dropped"] = comp["raw"] - comp["cleaned"]
comp["dropped %"] = (comp["dropped"] / comp["raw"] * 100).round(1).astype(str) + "%"

total = comp[["raw", "cleaned", "dropped"]].sum()
total["dropped %"] = str(round(total["dropped"] / total["raw"] * 100, 1)) + "%"
total.name = "TOTAL"
comp = pd.concat([comp, total.to_frame().T])
comp

,raw,cleaned,dropped,dropped %
24chasa,13178,12908,270,2.0%
actualno,7567,7241,326,4.3%
banker,998,997,1,0.1%
blitz,531,0,531,100.0%
bta,4652,4572,80,1.7%
capital,30,0,30,100.0%
dnevnik,256,0,256,100.0%
economic,466,466,0,0.0%
fakti,7640,7565,75,1.0%
monitor,13850,3403,10447,75.4%
